# Phase 0 — Notebook 01: Build trigger_events table

## Hypothesis (this notebook tests no edge hypothesis)

This notebook does **not** test any hypothesis about a betting edge. It does **not** compute features. Its sole job is to **materialize the research-side `trigger_events` artifacts** per `BUILD_SPEC.md` (V5/V5.1) with the addendum A.7 substitution: a clean, immutable, lookahead-safe enumeration of every "favorite trailing by D" first-occurrence event in regulation across the 10-season corpus, with the contextual fields needed for Notebooks **02a–g** to compute features **from `trigger_events.csv` only**, plus a sibling `trigger_outcomes.csv` for V5 LABEL columns (Notebook **03** joins both).

For each game in the working set defined by `data_quality_report.md`, this notebook detects the FIRST PLAY at which the pre-game favorite was trailing by `D` points, for `D ∈ {3, 7, 10, 14, 21}`, in regulation only (per A.2). Each first occurrence becomes one row in `trigger_events`, keyed by `UNIQUE(game_id, fav_deficit)`.

## What this notebook DOES NOT do

- Compute any feature (no `trigger_features`) — that's Notebooks 02a-g.
- Train any model — that's Notebook 03.
- Make any betting decisions — that's Phase 4.
- Create OT trigger rows — excluded entirely at ingest per addendum **A.2**.
- Filter out `(season, season_type)` pairs based on Elo coverage — N00 audit confirmed all 20 pairs cleared the 80% A.7 threshold (100% coverage across the board).

## Spec references

- `BUILD_SPEC.md` `trigger_events` DDL (full field list)
- `BUILD_SPEC.md` Owner addendum **A.2** — OT triggers excluded entirely at ingest
- `BUILD_SPEC.md` Owner addendum **A.7** — pre-game Elo replaces SP+/FPI
- `.cursorrules` rule **R3** — no lookahead bias; `assert_no_lookahead()` mandatory on every emitted row
- `.cursorrules` rule **R6** — three-table architecture; this notebook materializes the research-side `trigger_events` artifact (plus sibling `trigger_outcomes` for R3-safe label storage)
- `.cursorrules` rule **R15** — regulation only
- `.cursorrules` rule **R16** — `spread_movement` requires both opening AND closing spreads (NULL otherwise)
- `.cursorrules` rule **R22** — STOP at end of Notebook 01; do not start Notebook 02a without approval

## Deliverables produced by this notebook

1. `research/results/trigger_events.csv` — feature-safe context + in-game state **only** (committable; ~7700 rows × 45 cols). Notebooks **02a–g** load this file only — structurally impossible to leak post-game labels into features.
2. `research/results/trigger_outcomes.csv` — the V5 **OUTCOME / LABEL** block: `final_fav_won`, `final_fav_score`, `final_dog_score`, `margin_of_defeat`, keyed on `(game_id, fav_deficit)` (same row count as `trigger_events.csv`). Notebook **03** joins to this sibling on the natural key for targets.
3. `research/results/trigger_events_bucket_counts.csv` — per `(season, season_type, fav_deficit, quarter)` row counts (committable; anticipates Phase 0 acceptance gate 4)
4. `research/results/trigger_events.schema.md` — sidecar for `trigger_events.csv`: IDENTIFIER / PRE-GAME CONTEXT / POINT-IN-TIME GAME STATE; provider priority; empirical pre/post-play findings; generation date and source commit hash (committable)
5. `research/results/trigger_outcomes.schema.md` — sidecar for `trigger_outcomes.csv`: join key + the four LABEL columns (committable)

## Schema differences from BUILD_SPEC.md V5 `trigger_events` DDL

The `trigger_events` CSV is a superset-with-one-rename of the V5 DDL (section **`trigger_events` — what happened**):

- **Split to sibling file (4):** `final_fav_won`, `final_fav_score`, `final_dog_score`, `margin_of_defeat` → `trigger_outcomes.csv` (structural R3 guardrail; keeps feature extractors from ever seeing labels on the same DataFrame they parse).
- **Omitted (2):** `trigger_id` (auto-increment INTEGER PK is not meaningful in a CSV; the natural PK `(game_id, fav_deficit)` is enforced via UNIQUE check), `created_at` (provenance lives in the schema sidecars, not per-row).
- **Renamed (1):** `seconds_remaining` → `seconds_remaining_in_regulation` (disambiguates from per-quarter clock).
- **Semantics tightened (1):** `possession_team` stores the actual school name instead of the V5 enum `'favorite'|'underdog'`. `fav_has_ball` carries the favorite-perspective framing.
- **Added (17):** `season_type`, `home_team`, `away_team`, `home_is_fav`, `pregame_spread_provider`, `pregame_ml_provider`, `opening_spread_provider`, `play_number`, `play_type`, `clock_minutes_remaining`, `clock_seconds_remaining`, `clock_seconds_in_period_total`, `minutes_elapsed_total`, `actual_deficit_at_trigger`, `distance_to_first_down`, `down`, `drive_number_in_game`. Rationale per column is in `trigger_events.schema.md`.

Net column count: V5 DDL = 34, minus 4 LABEL (sibling CSV) minus 2 omitted = 28 DDL-adjacent columns in `trigger_events.csv`, plus 17 added = **45 columns**.

## Call budget

CFBD v2 free tier = 1000 calls/month. After Notebook 00: 87 used, 913 remaining this billing cycle.

This notebook is budgeted for **162 fresh `/plays` calls + 20 fresh `/drives` calls = 182 fresh CFBD calls**. Cache hits from Notebook 00 (`/games`, `/lines`, `/teams/fbs`, `/ratings/sp`, `/ratings/fpi`) are free. After this notebook: **269 / 1000** consumed, **731 remaining**. The final cell prints actual budget consumed.

In [ ]:
"""
Notebook 01 — imports, environment, path constants, fail-fast checks.
Same structure as Notebook 00. Run this cell first; if it raises, fix the
issue before continuing — none of the downstream cells will work without it.
"""
from __future__ import annotations

import csv
import hashlib
import json
import os
import pathlib
import subprocess
import time
from typing import Any

import httpx
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# --- Paths -------------------------------------------------------------------
NOTEBOOK_DIR = pathlib.Path(".").resolve()
RESEARCH_DIR = (NOTEBOOK_DIR / "..").resolve()
DATA_DIR = (RESEARCH_DIR / "data").resolve()
RESULTS_DIR = (RESEARCH_DIR / "results").resolve()
CACHE_DIR = DATA_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CALL_LOG = CACHE_DIR / "cfbd_call_log.csv"
ENV_PATH = (RESEARCH_DIR / ".." / "backend" / ".env").resolve()
REPO_ROOT = (RESEARCH_DIR / "..").resolve()

# --- Sanity check on workspace layout ----------------------------------------
assert RESEARCH_DIR.name == "research", (
    f"Expected to run inside research/notebooks/. Got NOTEBOOK_DIR={NOTEBOOK_DIR}. "
    f"cd into research/notebooks/ and re-launch jupyter."
)
assert ENV_PATH.exists(), (
    f"Did not find {ENV_PATH}. Per BUILD_SPEC.md A.5, the CFBD key lives in "
    f"backend/.env (NOT .env.example). Populate it before running this notebook."
)

# --- Load CFBD_API_KEY from backend/.env -------------------------------------
load_dotenv(ENV_PATH)

# --- FAIL FAST: refuse to run a single network call without the key ---------
assert os.environ.get("CFBD_API_KEY"), (
    "CFBD_API_KEY is not set. Expected to be populated in\n"
    f"    {ENV_PATH}\n"
    "Get a free key at https://collegefootballdata.com/key, then add\n"
    "    CFBD_API_KEY=your_key_here\n"
    "to backend/.env and re-run this cell. Calls 30 of 182 failing on 401\n"
    "wastes the entire monthly budget for nothing."
)

# --- Audit deliverable presence --------------------------------------------
# This notebook depends on Notebook 00's exclusion logic having been validated.
# We don't blindly trust audit_summary.csv — but its absence means Notebook 00
# was never run, in which case stop now.
AUDIT_SUMMARY_PATH = RESULTS_DIR / "audit_summary.csv"
assert AUDIT_SUMMARY_PATH.exists(), (
    f"Expected {AUDIT_SUMMARY_PATH} to exist (Notebook 00 deliverable). "
    f"Run Notebook 00 first, then this notebook."
)

print(f"[ok] paths resolved relative to {NOTEBOOK_DIR}")
print(f"[ok] CFBD_API_KEY loaded from {ENV_PATH}")
print(f"[ok] cache dir: {CACHE_DIR}")
print(f"[ok] audit_summary.csv present: {AUDIT_SUMMARY_PATH}")

In [ ]:
"""
HTTP helpers — same code as Notebook 00, same cache directory.
Cache hits (everything pulled in N00) cost zero from the call budget.

cfbd_get(endpoint, **params) -> JSON
om_get(**params) -> JSON

Cache key = sha1 of sorted JSON of params, truncated to 16 chars.
Every call (hit or miss, success or failure) is logged to
cache/cfbd_call_log.csv with timestamp, service, endpoint, params hash,
cached flag, status, response size, elapsed ms.
"""
CFBD_BASE = "https://apinext.collegefootballdata.com"
OPEN_METEO_BASE = "https://archive-api.open-meteo.com/v1/archive"

if not CALL_LOG.exists():
    with CALL_LOG.open("w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(
            ["timestamp", "service", "endpoint", "params_hash", "cached",
             "status", "bytes", "elapsed_ms"]
        )


def _params_hash(params: dict) -> str:
    return hashlib.sha1(json.dumps(params, sort_keys=True).encode()).hexdigest()[:16]


def _cache_key(prefix: str, params: dict) -> pathlib.Path:
    return CACHE_DIR / f"{prefix}__{_params_hash(params)}.json"


def _log(service: str, endpoint: str, params: dict, *, cached: bool,
         status: int, bytes_: int, elapsed_ms: int) -> None:
    with CALL_LOG.open("a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(
            [time.strftime("%Y-%m-%dT%H:%M:%S"), service, endpoint,
             _params_hash(params), int(cached), status, bytes_, elapsed_ms]
        )


def cfbd_get(endpoint: str, force_refresh: bool = False, **params: Any) -> Any:
    key = _cache_key(f"cfbd__{endpoint.strip('/').replace('/', '_')}", params)
    if key.exists() and not force_refresh:
        size = key.stat().st_size
        data = json.loads(key.read_text(encoding="utf-8"))
        _log("cfbd", endpoint, params, cached=True, status=200,
             bytes_=size, elapsed_ms=0)
        return data
    headers = {
        "Authorization": f"Bearer {os.environ['CFBD_API_KEY']}",
        "Accept": "application/json",
    }
    t0 = time.perf_counter()
    r = httpx.get(f"{CFBD_BASE}{endpoint}", params=params,
                  headers=headers, timeout=120)
    elapsed_ms = int((time.perf_counter() - t0) * 1000)
    _log("cfbd", endpoint, params, cached=False, status=r.status_code,
         bytes_=len(r.content), elapsed_ms=elapsed_ms)
    r.raise_for_status()
    data = r.json()
    key.write_text(json.dumps(data), encoding="utf-8")
    return data


def om_get(force_refresh: bool = False, **params: Any) -> Any:
    key = _cache_key("openmeteo__archive", params)
    if key.exists() and not force_refresh:
        size = key.stat().st_size
        data = json.loads(key.read_text(encoding="utf-8"))
        _log("open_meteo", "/v1/archive", params, cached=True, status=200,
             bytes_=size, elapsed_ms=0)
        return data
    t0 = time.perf_counter()
    r = httpx.get(OPEN_METEO_BASE, params=params, timeout=60)
    elapsed_ms = int((time.perf_counter() - t0) * 1000)
    _log("open_meteo", "/v1/archive", params, cached=False, status=r.status_code,
         bytes_=len(r.content), elapsed_ms=elapsed_ms)
    r.raise_for_status()
    data = r.json()
    key.write_text(json.dumps(data), encoding="utf-8")
    return data


print("[ok] cfbd_get and om_get defined")
print(f"[ok] sharing cache with Notebook 00 at {CACHE_DIR}")

## Configuration

`SEASONS` and `SEASON_TYPES` match Notebook 00. `DEFICIT_THRESHOLDS` are the five values from the V5/V5.1 schema. `PROVIDER_PRIORITY_REAL_BOOKS` is **alphabetically ordered**: `Bovada`, `Caesars`, `DraftKings`, `ESPN Bet`. The N00 audit printed a per-(season, season_type) provider count matrix, but that matrix was not written to a checked-in CSV, so per-provider coverage cannot be cited from project artifacts at this point. Re-evaluate the ordering in Notebook 03 if walk-forward results show ordering affects calibration or CLV. `consensus` is a **fallback** for spread-only fields when no real book has data — it is excluded from moneyline aggregation entirely (it does not represent a real, bettable price). `numberfire` and `teamrankings` are rating-service projections, not market prices, and are **never** used.

The provider choice is documented at run time in `trigger_events.schema.md` (sidecar) so future readers do not re-litigate it.

In [ ]:
SEASONS: list[int] = list(range(2015, 2025))
SEASON_TYPES: list[str] = ["regular", "postseason"]

# Deficit thresholds from BUILD_SPEC.md trigger_events.fav_deficit field.
DEFICIT_THRESHOLDS: list[int] = [3, 7, 10, 14, 21]

# trigger_features.feature_set_version follows trigger_events.feature_set_version
# (R6 separation), but trigger_events itself is keyed by (game_id, fav_deficit)
# only — the V5 spec does not version trigger_events. We tag the run for
# provenance and so downstream notebooks can assert they are reading what
# they expect.
TRIGGER_TABLE_VERSION: str = "v1"

# --- Provider priority (see preceding markdown cell for rationale) ---------
# Alphabetical ordering. Per-provider coverage was not computed against
# a checked-in artifact, so any coverage-weighted reordering is deferred
# to Notebook 03 if walk-forward results show ordering affects metrics.
PROVIDER_PRIORITY_REAL_BOOKS: list[str] = [
    "Bovada",
    "Caesars",
    "DraftKings",
    "ESPN Bet",
]
SPREAD_FALLBACK_PROVIDER: str = "consensus"  # spread-only fallback
EXCLUDED_PROVIDERS: set[str] = {"numberfire", "teamrankings"}  # rating projections, not markets

# --- A.7 Elo coverage threshold from N00 audit ----------------------------
# All 20 (season, season_type) pairs cleared this in N00; we re-assert here
# instead of re-computing.
ELO_THRESHOLD_PCT: float = 80.0

print(f"seasons: {SEASONS}")
print(f"season types: {SEASON_TYPES}")
print(f"deficit thresholds: {DEFICIT_THRESHOLDS}")
print(f"trigger table version: {TRIGGER_TABLE_VERSION}")
print(f"provider priority (ML + spread): {PROVIDER_PRIORITY_REAL_BOOKS}")
print(f"spread-only fallback: {SPREAD_FALLBACK_PROVIDER}")
print(f"excluded providers: {sorted(EXCLUDED_PROVIDERS)}")

## Phase 0a — Re-load cached metadata + apply audit exclusions

Re-pull `/games` and `/lines` from the cache (free; populated by Notebook 00). Re-apply N00's per-game exclusion logic so the working set is exactly what audit_summary.csv documented. Then determine the favorite per game and aggregate provider lines into the canonical pre-game fields.

In [ ]:
# --- Re-load /games (cache hits — zero budget cost) -----------------------
games_records: list[dict] = []
for year in SEASONS:
    for season_type in SEASON_TYPES:
        games = cfbd_get(
            "/games",
            year=year,
            seasonType=season_type,
            classification="fbs",
        )
        for g in games:
            g["_audit_season"] = year
            g["_audit_season_type"] = season_type
            games_records.append(g)
games_df = pd.json_normalize(games_records)
print(f"[ok] games loaded: {len(games_df)} rows (cached)")

# --- Re-load /lines (cache hits) ------------------------------------------
lines_records: list[dict] = []
for year in SEASONS:
    for season_type in SEASON_TYPES:
        lines = cfbd_get(
            "/lines",
            year=year,
            seasonType=season_type,
            classification="fbs",
        )
        for entry in lines:
            entry["_audit_season"] = year
            entry["_audit_season_type"] = season_type
            lines_records.append(entry)
print(f"[ok] lines loaded: {len(lines_records)} entries (cached)")

# Build a quick lookup: game_id -> list of provider line dicts.
lines_by_game: dict[int, list[dict]] = {}
for entry in lines_records:
    gid = entry.get("id")
    if gid is None:
        continue
    lines_by_game[gid] = entry.get("lines") or []
print(f"[ok] lines_by_game lookup: {len(lines_by_game)} games")

In [ ]:
# --- Per-game flags (same semantics as N00) -------------------------------
def _classify(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value.lower()
    if isinstance(value, dict):
        return str(value.get("name") or value.get("value") or "").lower()
    return str(value).lower()


def _line_scores_len(value: Any) -> int | None:
    if isinstance(value, list):
        return len(value)
    return None


games_df["home_class"] = games_df["homeClassification"].apply(_classify)
games_df["away_class"] = games_df["awayClassification"].apply(_classify)
games_df["fbs_vs_fbs"] = (games_df["home_class"] == "fbs") & (games_df["away_class"] == "fbs")
games_df["home_ls_len"] = games_df["homeLineScores"].apply(_line_scores_len)
_completed_bool = games_df["completed"].fillna(False).astype(bool)
games_df["shortened_or_canceled"] = (
    (~_completed_bool)
    | (_completed_bool & games_df["home_ls_len"].fillna(0).astype(int).lt(4))
)

# Per-game has-spread flag — uses provider priority (real books + consensus
# fallback). At this stage we just need a boolean: at least one acceptable
# provider has a spread for this game.
def _game_has_acceptable_spread(game_id: int) -> bool:
    for line in lines_by_game.get(game_id, []):
        prov = (line.get("provider") or "").strip()
        if prov in EXCLUDED_PROVIDERS:
            continue
        if line.get("spread") is not None:
            return True
    return False


games_df["has_acceptable_spread"] = games_df["id"].apply(_game_has_acceptable_spread)

# Final per-game exclusion filter (matches N00 logic with one tweak: we use
# our own provider whitelist rather than "any provider has a spread").
games_df["passes_exclusion_filter"] = (
    _completed_bool
    & games_df["fbs_vs_fbs"]
    & games_df["has_acceptable_spread"]
    & ~games_df["shortened_or_canceled"]
)

n_total = len(games_df)
n_after = int(games_df["passes_exclusion_filter"].sum())
print(f"games before exclusion: {n_total}")
print(f"games after exclusion: {n_after}")
print(f"  excluded for not-completed:    {int((~_completed_bool).sum())}")
print(f"  excluded for FCS opponent:     {int((~games_df['fbs_vs_fbs']).sum())}")
print(f"  excluded for shortened/canceled: {int(games_df['shortened_or_canceled'].sum())}")
print(f"  excluded for no acceptable spread: {int((~games_df['has_acceptable_spread']).sum())}")

# Cross-check against audit_summary.csv (sanity).
audit_summary_df = pd.read_csv(AUDIT_SUMMARY_PATH)
audit_n_after = int(audit_summary_df["n_games_after_exclusions"].sum())
print(f"\naudit_summary.csv n_games_after_exclusions sum: {audit_n_after}")
if abs(audit_n_after - n_after) > 5:
    # Small mismatch is expected because N00's exclusion used "any non-null spread"
    # while we use the real-book whitelist. >5 mismatch is suspicious.
    print(f"[note] working-set size differs from N00 by {audit_n_after - n_after}; "
          f"this is expected because we use a stricter provider whitelist.")

In [ ]:
# --- Determine favorite per game from priority-aggregated spread ----------
# Convention: pregame_spread is ALWAYS <= 0 (favorite negative). For a 7-pt
# favorite, pregame_spread = -7. For a pick'em (home_spread == 0), the game
# is excluded entirely — no nominal-favorite assignment. <1% of games.
def _aggregate_spread(game_id: int, field: str = "spread") -> tuple[float | None, str | None]:
    """Return (value, provider_used) for the highest-priority provider with
    a non-null value for `field`. Tries real books in order, then consensus
    as fallback (for spread fields only — consensus is excluded for ML).
    """
    lines = lines_by_game.get(game_id, [])
    by_provider: dict[str, dict] = {(line.get("provider") or "").strip(): line for line in lines}
    for prov in PROVIDER_PRIORITY_REAL_BOOKS:
        line = by_provider.get(prov)
        if line is None:
            continue
        v = line.get(field)
        if v is not None:
            return float(v), prov
    # Spread-only fallback to consensus.
    if field in ("spread", "spreadOpen"):
        line = by_provider.get(SPREAD_FALLBACK_PROVIDER)
        if line is not None and line.get(field) is not None:
            return float(line[field]), SPREAD_FALLBACK_PROVIDER
    return None, None


def _aggregate_ml(game_id: int, field: str) -> tuple[float | None, str | None]:
    """ML aggregation — REAL BOOKS ONLY. consensus is excluded because it
    does not represent a tradeable price; aggregator MLs have inconsistent
    vig and break devigging math in N04.
    """
    lines = lines_by_game.get(game_id, [])
    by_provider: dict[str, dict] = {(line.get("provider") or "").strip(): line for line in lines}
    for prov in PROVIDER_PRIORITY_REAL_BOOKS:
        line = by_provider.get(prov)
        if line is None:
            continue
        v = line.get(field)
        if v is not None:
            return float(v), prov
    return None, None


# Iterate working-set games and compute pre-game fields per row.
working_set_df = games_df[games_df["passes_exclusion_filter"]].copy()
print(f"working-set games: {len(working_set_df)}")

pregame_rows: list[dict] = []
n_pickem_excluded = 0
for _, g in working_set_df.iterrows():
    gid = int(g["id"])
    home_spread, spread_provider = _aggregate_spread(gid, "spread")
    if home_spread is None:
        # Should be impossible given has_acceptable_spread, but guard anyway.
        continue
    if home_spread == 0:
        n_pickem_excluded += 1
        continue

    home_is_fav = home_spread < 0
    fav_team = g["homeTeam"] if home_is_fav else g["awayTeam"]
    dog_team = g["awayTeam"] if home_is_fav else g["homeTeam"]
    pregame_spread = -abs(home_spread)  # always <= 0

    home_pe = g.get("homePregameElo")
    away_pe = g.get("awayPregameElo")
    if pd.notna(home_pe) and pd.notna(away_pe):
        fav_pregame_rating = float(home_pe if home_is_fav else away_pe)
        dog_pregame_rating = float(away_pe if home_is_fav else home_pe)
        rating_gap = fav_pregame_rating - dog_pregame_rating
    else:
        fav_pregame_rating = None
        dog_pregame_rating = None
        rating_gap = None

    home_ml, ml_provider = _aggregate_ml(gid, "homeMoneyline")
    away_ml, _ = _aggregate_ml(gid, "awayMoneyline")
    pregame_fav_ml = (home_ml if home_is_fav else away_ml) if (home_ml is not None or away_ml is not None) else None
    pregame_dog_ml = (away_ml if home_is_fav else home_ml) if (home_ml is not None or away_ml is not None) else None

    home_open, open_provider = _aggregate_spread(gid, "spreadOpen")
    if home_open is not None:
        opening_spread = -abs(home_open)
        home_open_was_fav = home_open < 0
        # Sign convention check: opening_spread expressed in same fav-perspective
        # as pregame_spread. If favorite changed between opening and closing
        # (rare; flagged below), opening_spread reflects the closing-favorite's
        # opening spread, even if that means it was positive at open. We carry
        # the magnitude with the closing favorite's sign.
        opening_spread_signed_to_closing_fav = home_open if home_is_fav else -home_open
    else:
        opening_spread = None
        home_open_was_fav = None
        opening_spread_signed_to_closing_fav = None

    closing_spread = pregame_spread  # closing == pregame for pre-kickoff
    closing_spread_signed_to_closing_fav = home_spread if home_is_fav else -home_spread

    # Per R16, spread_movement is NULL when either opening or closing is
    # missing. Movement is in points, sign convention "favorite negative":
    #   spread_movement = closing_spread - opening_spread
    # Negative movement = favorite getting more favored (line moved against dog).
    # Positive movement = favorite getting less favored (line moved against fav).
    if opening_spread is None or closing_spread is None:
        spread_movement = None
        line_moved_against_fav = None
    else:
        spread_movement = closing_spread_signed_to_closing_fav - opening_spread_signed_to_closing_fav
        line_moved_against_fav = bool(spread_movement > 0)

    # Outcome / label fields per V5 DDL ("OUTCOME — filled when game ends").
    # All games in the working set are completed, so we can populate now.
    final_home_score = int(g.get("homePoints")) if pd.notna(g.get("homePoints")) else None
    final_away_score = int(g.get("awayPoints")) if pd.notna(g.get("awayPoints")) else None
    if final_home_score is not None and final_away_score is not None:
        final_fav_score = final_home_score if home_is_fav else final_away_score
        final_dog_score = final_away_score if home_is_fav else final_home_score
        if final_fav_score > final_dog_score:
            final_fav_won = True
            margin_of_defeat = None
        elif final_fav_score < final_dog_score:
            final_fav_won = False
            margin_of_defeat = int(final_dog_score - final_fav_score)
        else:
            # CFB ties don't really happen post-OT but guard anyway.
            final_fav_won = None
            margin_of_defeat = None
    else:
        final_fav_score = None
        final_dog_score = None
        final_fav_won = None
        margin_of_defeat = None

    pregame_rows.append({
        "game_id": gid,
        "season": int(g["_audit_season"]),
        "season_type": str(g["_audit_season_type"]),
        "week": int(g.get("week")) if pd.notna(g.get("week")) else None,
        "fav_team": fav_team,
        "dog_team": dog_team,
        "home_team": g["homeTeam"],
        "away_team": g["awayTeam"],
        "home_is_fav": home_is_fav,
        "pregame_spread": pregame_spread,
        "pregame_spread_provider": spread_provider,
        "pregame_fav_ml": pregame_fav_ml,
        "pregame_dog_ml": pregame_dog_ml,
        "pregame_ml_provider": ml_provider,
        "fav_pregame_rating": fav_pregame_rating,
        "dog_pregame_rating": dog_pregame_rating,
        "rating_gap": rating_gap,
        "opening_spread": opening_spread,
        "opening_spread_provider": open_provider,
        "closing_spread": closing_spread,
        "spread_movement": spread_movement,
        "line_moved_against_fav": line_moved_against_fav,
        "final_home_score": final_home_score,
        "final_away_score": final_away_score,
        "final_fav_won": final_fav_won,
        "final_fav_score": final_fav_score,
        "final_dog_score": final_dog_score,
        "margin_of_defeat": margin_of_defeat,
    })

pregame_df = pd.DataFrame(pregame_rows).set_index("game_id", drop=True)
print(f"working-set games after pick'em exclusion: {len(pregame_df)}")
print(f"games excluded as pick'ems (home spread == 0): {n_pickem_excluded}")
print(f"\nprovider mix actually selected (closing spread):")
print(pregame_df["pregame_spread_provider"].value_counts(dropna=False).to_string())
print(f"\nprovider mix actually selected (opening spread):")
print(pregame_df["opening_spread_provider"].value_counts(dropna=False).to_string())
print(f"\nprovider mix actually selected (moneyline):")
print(pregame_df["pregame_ml_provider"].value_counts(dropna=False).to_string())

## Phase 0b — Fetch play-by-play and drive metadata

This is the network-heavy section. `/plays` is paginated by `(year, seasonType, week)` (162 distinct tuples in our working set) and `/drives` is paginated by `(year, seasonType)` (20 tuples). All calls go through the cache so re-runs are instant.

In [ ]:
# --- Determine the (season, season_type, week) tuples we actually need ---
# Use the pregame_df working set (post-exclusion) so we never call /plays for
# (year, type, week) combos that have zero working-set games.
work_tuples_df = (
    pregame_df[["season", "season_type", "week"]]
    .drop_duplicates()
    .sort_values(["season", "season_type", "week"])
    .reset_index(drop=True)
)
print(f"distinct (season, season_type, week) tuples to pull: {len(work_tuples_df)}")
work_tuples_df.head(5)

In [ ]:
# --- Fetch /plays for every (season, season_type, week) tuple ------------
# Total: 162 tuples on first run -> 162 fresh /plays calls. Re-runs are zero
# cost. Each tuple returns ~1500-3000 plays for regular weeks, fewer for
# postseason.
plays_by_game: dict[int, list[dict]] = {}
plays_pull_log: list[dict] = []
t_start = time.perf_counter()
for i, row in work_tuples_df.iterrows():
    season = int(row["season"])
    season_type = str(row["season_type"])
    week = int(row["week"])
    plays = cfbd_get(
        "/plays",
        year=season,
        seasonType=season_type,
        week=week,
        classification="fbs",
    )
    n_plays = len(plays)
    n_games_in_pull = len({p.get("gameId") for p in plays if p.get("gameId") is not None})
    plays_pull_log.append({
        "season": season,
        "season_type": season_type,
        "week": week,
        "n_plays": n_plays,
        "n_games_in_pull": n_games_in_pull,
    })
    for p in plays:
        gid = p.get("gameId")
        if gid is None:
            continue
        plays_by_game.setdefault(gid, []).append(p)
    if (i + 1) % 30 == 0 or (i + 1) == len(work_tuples_df):
        elapsed = time.perf_counter() - t_start
        print(f"  pulled {i+1:>3}/{len(work_tuples_df)} tuples in {elapsed:6.1f}s "
              f"(running total: {sum(x['n_plays'] for x in plays_pull_log):,} plays "
              f"across {len({gid for tup in plays_pull_log for gid in plays_by_game})} games)")

print(f"\n[ok] /plays pull complete: {sum(x['n_plays'] for x in plays_pull_log):,} total plays "
      f"across {len(plays_by_game):,} games")

In [ ]:
# --- Fetch /drives for every (season, season_type) -----------------------
# Total: 20 tuples on first run -> 20 fresh /drives calls. /drives returns
# all FBS drives for the (year, type) pair in a single response.
drives_by_game: dict[int, list[dict]] = {}
for year in SEASONS:
    for season_type in SEASON_TYPES:
        drives = cfbd_get(
            "/drives",
            year=year,
            seasonType=season_type,
            classification="fbs",
        )
        for d in drives:
            gid = d.get("gameId")
            if gid is None:
                continue
            drives_by_game.setdefault(gid, []).append(d)

print(f"[ok] /drives pull complete: {sum(len(v) for v in drives_by_game.values()):,} drives "
      f"across {len(drives_by_game):,} games")

### Empirical verification of `SCORING_PLAY_REGISTRY`

CFBD's `offenseScore` / `defenseScore` semantics are not documented in the v2 docs we have access to, and an earlier 3-play-type check (Touchdown, Field Goal, Safety) returned a `MIXED` verdict that masked three distinct parsing bugs (return-TD attribution, duplicate `playNumber` rows, paired-play recording). To replace the `MIXED` silent-fallback with explicit per-subtype handling, we now:

1. **Hand-curate** the set of `playType` strings that may carry `scoring=true` (`KNOWN_SCORING_PLAYTYPES` below). Any `scoring=true` non-PAT play whose `playType` is not in the curated set raises here -- so a future CFBD addition cannot silently land in `trigger_events` under POST_PLAY defaults.
2. **Sort** plays by `(period, driveNumber, playNumber)`, the only triple that totally orders plays in every game in the cached corpus, and **deduplicate** same-key collisions by keeping the play with the lowest CFBD `id`.
3. **Verify** each curated subtype empirically by triangulating `(this_score - prev_clean_score)` and `(next_clean_score - this_score)` against the candidate `(side, points)` pair, and assign one of three confidence tiers:

   - `POST_PLAY` (>= `MIN_SAMPLES` AND >= `VERIFIED_POST_THRESHOLD` POST share)
   - `POST_BEST_EFFORT` (KO Return TD carve-out at `KO_RETURN_TD_THRESHOLD`; +/-2 cross-check at scorer time)
   - `BEST_EFFORT_LOW_N` (< `MIN_SAMPLES`; +/-7 cross-check at scorer time)

The full per-subtype tier table is documented in `trigger_events.schema.md`. Verdicts persist to `_subtype_verdicts.json` so future N01 re-runs can warn (not error) on >2pp drift in any subtype's POST share.

In [ ]:
# --- Empirical verification of SCORING_PLAY_REGISTRY ---------------------
# This cell builds the per-subtype score-state registry that the trigger
# detection scorer uses in cell 16. Two design decisions, both tightened
# from the original (broken) version:
#
#   1. The set of playType strings that may have scoring=true is HAND-
#      CURATED (KNOWN_SCORING_PLAYTYPES below). Cell 14 verifies each
#      curated subtype empirically against the cached corpus and assigns
#      it a verdict; it does NOT discover subtypes at runtime. Any
#      scoring=true non-PAT play whose playType is not in the curated
#      set raises here, so a future CFBD addition cannot silently land
#      in trigger_events under POST_PLAY defaults.
#
#   2. Plays are sorted by (period, driveNumber, playNumber) — the only
#      triple that totally orders plays in every game in the cached
#      corpus. playNumber alone is per-drive (so it collides across
#      drives within a period) and CFBD's `id` is mostly but not
#      reliably monotonic across drives. Same-key duplicates are
#      deduplicated by keeping the play with the lowest CFBD id.
#
# Verdict tiers (used by cell 16's scorer; documented in
# trigger_events.schema.md):
#   POST_PLAY            -- ≥MIN_SAMPLES samples and ≥VERIFIED_POST_THRESHOLD
#                           POST share. Scorer reads offenseScore/defenseScore
#                           direct.
#   POST_BEST_EFFORT     -- KO Return TD carve-out: ≥KO_RETURN_TD_THRESHOLD
#                           POST share. Scorer applies a ±2 cross-check at
#                           use time.
#   BEST_EFFORT_LOW_N    -- <MIN_SAMPLES samples. Scorer applies a ±7
#                           cross-check at use time. Soft guarantee;
#                           downstream notebooks should treat triggers from
#                           these subtypes with appropriate skepticism.
#   FAIL / FAIL_NO_SAMPLES -- below threshold or never observed. Cell 14
#                           raises here (does not let cell 16 silently use
#                           the value).

from collections import Counter, defaultdict

VERIFIED_POST_THRESHOLD = 0.95
KO_RETURN_TD_THRESHOLD = 0.65
MIN_SAMPLES = 20

KNOWN_SCORING_PLAYTYPES: set[str] = {
    # --- Touchdown subtypes ---
    "rushing touchdown", "passing touchdown",
    "interception return touchdown", "fumble return touchdown",
    "punt return touchdown", "kickoff return touchdown",
    "blocked punt touchdown", "blocked field goal touchdown",
    "missed field goal return touchdown",
    # --- Field goal + safety ---
    "field goal good", "safety",
    # --- CFBD records some scoring plays under non-touchdown playTypes
    #     because the play started as something else (a punt that was
    #     blocked and returned, an interception that was returned, etc.).
    #     Each one is empirically verified below; the scorer treats them
    #     via verdict only (no semantic side/points used at scorer time).
    "fumble recovery (opponent)", "fumble recovery (own)",
    "interception", "sack",
    "punt", "kickoff", "kickoff return (offense)",
    "rush", "pass reception",
    "blocked punt", "blocked field goal",
    "pass interception return", "pass incompletion",
    "penalty", "timeout", "uncategorized",
}

# PAT-like detection -- broader than the original (which only matched
# substrings on playType). PAT plays do not fire triggers independently;
# their points are already credited under POST_PLAY by the prior TD's
# scoreboard.
_PAT_PLAYTYPE_KEYWORDS = ("point after", "two point conversion", "extra point")
_PAT_PLAYTYPE_EXACT = {
    "two point pass", "two point rush", "defensive 2pt conversion",
}
_PAT_PLAYTEXT_KEYWORDS = ("extra point", "two-point", "two point", "defensive pat")


def _is_pat_like(play: dict) -> bool:
    pt = (play.get("playType") or "").strip().lower()
    if any(kw in pt for kw in _PAT_PLAYTYPE_KEYWORDS):
        return True
    if pt in _PAT_PLAYTYPE_EXACT:
        return True
    txt = (play.get("playText") or "").strip().lower()
    if any(kw in txt for kw in _PAT_PLAYTEXT_KEYWORDS):
        return True
    return False


def _chronological_sort_key(p: dict) -> tuple[int, int, int]:
    """Canonical chronological order of plays within a game."""
    return (int(p["period"]), int(p["driveNumber"]), int(p["playNumber"]))


def _read_team_score(play: dict, team: str) -> int | None:
    if play.get("offense") == team:
        v = play.get("offenseScore")
        return int(v) if v is not None else None
    if play.get("defense") == team:
        v = play.get("defenseScore")
        return int(v) if v is not None else None
    return None


def _find_clean_prev(plays_sorted: list[dict], i: int, team: str,
                     this_score: int) -> dict | None:
    """Walk backward to the first non-PAT play where the team's score
    differs from `this_score`. Skipping same-score plays handles paired-
    play recording, where the row immediately before a TD already
    carries the post-TD scoreboard (e.g., a `Rush` row right before a
    `Rushing Touchdown` row, both reading 21-14)."""
    for j in range(i - 1, -1, -1):
        prev = plays_sorted[j]
        if _is_pat_like(prev):
            continue
        ps = _read_team_score(prev, team)
        if ps is None:
            return None
        if ps == this_score:
            continue
        return prev
    return None


def _find_clean_next(plays_sorted: list[dict], i: int, team: str) -> dict | None:
    """Walk forward to the first non-PAT play with a readable score for
    `team`. Used for triangulating POST vs PRE convention here, and for
    the ±2 / ±7 cross-checks applied at scorer time."""
    for j in range(i + 1, len(plays_sorted)):
        nxt = plays_sorted[j]
        if _is_pat_like(nxt):
            continue
        ns = _read_team_score(nxt, team)
        if ns is None:
            return None
        return nxt
    return None


def _sorted_plays_for_game(plays: list[dict]) -> tuple[list[dict], int]:
    """Sort + dedup plays for a single game.

    Returns (sorted_plays, n_discards). Drops plays missing any of
    period/driveNumber/playNumber. Within (period, driveNumber, playNumber)
    duplicates, keeps the play with the lowest CFBD `id` and discards the
    rest. Used by trigger detection (cell 16), assert_no_lookahead
    (cell 17), and the empirical verification below."""
    valid = [p for p in plays
             if p.get("period") is not None
             and p.get("driveNumber") is not None
             and p.get("playNumber") is not None]
    valid.sort(key=_chronological_sort_key)
    seen: dict[tuple[int, int, int], dict] = {}
    discards = 0
    for p in valid:
        k = _chronological_sort_key(p)
        if k in seen:
            existing_id = int(seen[k].get("id") or (1 << 62))
            challenger_id = int(p.get("id") or (1 << 62))
            if challenger_id < existing_id:
                seen[k] = p
            discards += 1
        else:
            seen[k] = p
    return sorted(seen.values(), key=_chronological_sort_key), discards


# --- Pre-compute deduplicated, chronologically sorted plays per game ---
SORT_DEDUP_LOG: list[dict] = []
sorted_plays_cache: dict[int, list[dict]] = {}
for gid, plays in plays_by_game.items():
    sp, discards = _sorted_plays_for_game(plays)
    if discards:
        # Hard-fail if a single game has more than 3% duplicates (or
        # more than 3 events in total) -- looks like a different data
        # quality issue than the documented Flavor-A / Flavor-B
        # patterns we have evidence for in the corpus.
        if discards > max(3, 3 * len(plays) // 100):
            raise AssertionError(
                f"game_id={gid}: {discards} duplicate (period, driveNumber, "
                f"playNumber) rows out of {len(plays)} plays. Suspicious data "
                f"quality; investigate before continuing."
            )
        SORT_DEDUP_LOG.append({"game_id": gid, "discards": discards,
                               "n_plays": len(plays)})
    sorted_plays_cache[gid] = sp

print(f"[ok] sorted_plays_cache built for {len(sorted_plays_cache):,} games")
print(f"     dedup events: {len(SORT_DEDUP_LOG):,} games "
      f"({sum(e['discards'] for e in SORT_DEDUP_LOG):,} rows discarded total)")


# --- Empirical verification per subtype --------------------------------
# For each curated subtype, scan the corpus and triangulate POST vs PRE
# convention by reading (this play's score - prev clean play's score) and
# (next clean play's score - this play's score). The (side, points) pair
# is an internal verification key: we sweep all combinations and pick the
# one with the highest VERIFIED_POST share. The (side, points) is NOT a
# semantic football attribution -- the scorer in cell 16 reads scores
# direct via offenseScore/defenseScore lookup, and only consults the
# verdict.
print("\nVerifying SCORING_PLAY_REGISTRY against cached corpus...")
SUBTYPE_TRIAL_RESULTS: dict[str, dict[tuple[str, int], Counter]] = defaultdict(
    lambda: defaultdict(Counter)
)

# Coverage: any scoring=true non-PAT playType not in KNOWN_SCORING_PLAYTYPES
# is a curated-set bug; we surface it explicitly instead of silently
# skipping.
unknown_subtypes: Counter = Counter()

for gid, sp in sorted_plays_cache.items():
    for i, p in enumerate(sp):
        if not bool(p.get("scoring")):
            continue
        if _is_pat_like(p):
            continue
        if int(p.get("period") or 0) >= 5:
            continue
        pt_l = (p.get("playType") or "").strip().lower()
        if not pt_l:
            unknown_subtypes["(empty playType)"] += 1
            continue
        if pt_l not in KNOWN_SCORING_PLAYTYPES:
            unknown_subtypes[pt_l] += 1
            continue

        for side in ("offense", "defense"):
            team = p.get(side)
            if not team:
                continue
            ts = _read_team_score(p, team)
            if ts is None:
                continue
            prev = _find_clean_prev(sp, i, team, ts)
            if prev is None:
                continue
            ps = _read_team_score(prev, team)
            nxt = _find_clean_next(sp, i, team)
            if nxt is None:
                continue
            ns = _read_team_score(nxt, team)
            d_pre = ts - ps
            d_next = ns - ts
            for points in (6, 3, 2):
                if d_pre >= points and d_next <= 2:
                    SUBTYPE_TRIAL_RESULTS[pt_l][(side, points)]["VERIFIED_POST"] += 1
                elif d_pre == 0 and d_next >= points:
                    SUBTYPE_TRIAL_RESULTS[pt_l][(side, points)]["VERIFIED_PRE"] += 1

if unknown_subtypes:
    raise AssertionError(
        f"Cached corpus contains scoring=true non-PAT plays with playTypes "
        f"not in KNOWN_SCORING_PLAYTYPES:\n  "
        + "\n  ".join(f"{k!r}: {v:,} plays" for k, v in unknown_subtypes.most_common())
        + "\nAdd them to KNOWN_SCORING_PLAYTYPES (and re-run the empirical "
          "verification) before re-building trigger_events. Silent fallback "
          "to POST_PLAY would land bad rows in N02a-g and N03 input."
    )


# --- Pick best (side, points) per subtype, classify verdict -----------
SCORING_PLAY_REGISTRY: dict[str, dict] = {}
for pt_l in sorted(KNOWN_SCORING_PLAYTYPES):
    trials = SUBTYPE_TRIAL_RESULTS.get(pt_l, {})
    best_share = -1.0
    best_n = 0
    best_post = 0
    best_pre = 0
    best_side: str | None = None
    best_points: int | None = None
    for (side, points), c in trials.items():
        n = c["VERIFIED_POST"] + c["VERIFIED_PRE"]
        if n < 1:
            continue
        share = c["VERIFIED_POST"] / n
        if (share > best_share) or (share == best_share and n > best_n):
            best_share = share
            best_n = n
            best_post = c["VERIFIED_POST"]
            best_pre = c["VERIFIED_PRE"]
            best_side = side
            best_points = points

    if best_n == 0:
        verdict = "FAIL_NO_SAMPLES"
    elif best_n < MIN_SAMPLES:
        verdict = "BEST_EFFORT_LOW_N"
    elif pt_l == "kickoff return touchdown":
        verdict = ("POST_BEST_EFFORT" if best_share >= KO_RETURN_TD_THRESHOLD
                   else "FAIL")
    else:
        verdict = ("POST_PLAY" if best_share >= VERIFIED_POST_THRESHOLD
                   else "FAIL")

    SCORING_PLAY_REGISTRY[pt_l] = {
        "verdict": verdict,
        "n_samples": best_n,
        "verified_post": best_post,
        "verified_pre": best_pre,
        "post_share": (best_share if best_share >= 0 else None),
        "_side": best_side,         # internal triangulation key, NOT semantic
        "_points": best_points,     # internal triangulation key, NOT semantic
    }


# --- Print + gate ------------------------------------------------------
print(f"\nSCORING_PLAY_REGISTRY entries: {len(SCORING_PLAY_REGISTRY)}")
print(f"\n  {'playType':<40} {'n':>6}  POST%   verdict")
fails: list[str] = []
for pt_l in sorted(SCORING_PLAY_REGISTRY.keys()):
    r = SCORING_PLAY_REGISTRY[pt_l]
    pct_str = f"{100 * r['post_share']:5.1f}%" if r["post_share"] is not None else "  --  "
    print(f"  {pt_l:<40} {r['n_samples']:>6,}  {pct_str}  {r['verdict']}")
    if r["verdict"] in ("FAIL", "FAIL_NO_SAMPLES"):
        fails.append(pt_l)

if fails:
    raise AssertionError(
        "Empirical verification failed for the following subtypes; "
        "scorer would raise on every encounter, so we stop here instead "
        "of producing partial output:\n  "
        + "\n  ".join(fails)
    )

n_post = sum(1 for r in SCORING_PLAY_REGISTRY.values() if r["verdict"] == "POST_PLAY")
n_be   = sum(1 for r in SCORING_PLAY_REGISTRY.values() if r["verdict"] == "POST_BEST_EFFORT")
n_low  = sum(1 for r in SCORING_PLAY_REGISTRY.values() if r["verdict"] == "BEST_EFFORT_LOW_N")
print(f"\nTier summary:")
print(f"  VERIFIED_POST_PLAY  : {n_post}")
print(f"  POST_BEST_EFFORT    : {n_be}")
print(f"  BEST_EFFORT_LOW_N   : {n_low}")
print(f"  FAIL / FAIL_NO_SAMPLES: 0  (gated above)")

# Convention banner. Kept for backward compatibility (cell 17 still reads
# this); the scorer in cell 16 now branches on registry verdict, not on
# this string.
SCORE_STATE_CONVENTION = "MIXED_BY_PLAY_TYPE_VERIFIED"
print(f"\nSCORE_STATE_CONVENTION = {SCORE_STATE_CONVENTION!r}")

# Persist the empirical findings to a sidecar so future runs can diff
# and warn (not error) on >2pp drift in any subtype's POST share.
SUBTYPE_VERDICTS_PATH = RESULTS_DIR / "_subtype_verdicts.json"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
_persist = {pt_l: {k: v for k, v in r.items() if not k.startswith("_")}
            for pt_l, r in SCORING_PLAY_REGISTRY.items()}
SUBTYPE_VERDICTS_PATH.write_text(
    json.dumps({"thresholds": {"verified_post": VERIFIED_POST_THRESHOLD,
                               "ko_return_td":  KO_RETURN_TD_THRESHOLD,
                               "min_samples":   MIN_SAMPLES},
                "score_state_convention": SCORE_STATE_CONVENTION,
                "subtypes": _persist},
               indent=2, sort_keys=True),
    encoding="utf-8",
)
print(f"[ok] wrote {SUBTYPE_VERDICTS_PATH.relative_to(REPO_ROOT)}")

## Phase 0c — Trigger detection

`detect_triggers_for_game` walks plays in chronological order and emits, for each deficit threshold `D`, the FIRST regulation play after which the favorite was trailing by at least `D` points. OT plays (`period >= 5`) are excluded entirely per A.2/R15. Within a single game, triggers are ordered chronologically (with ascending `D` as tiebreak when multiple thresholds cross on the same play) and assigned `trigger_sequence` 1, 2, 3, ...

`assert_no_lookahead` is the per-row hard gate: every emitted row must satisfy `play.playNumber == trigger.play_number` AND every `point_in_time_field` must be derivable from plays with `playNumber <= trigger.play_number` only.

In [ ]:
class ScoringPlayAmbiguityError(ValueError):
    """Raised by _post_play_scores_for_team when the scorer cannot
    certify the post-play score for a scoring play. The caller must
    decide whether the ambiguity is (a) a documented cross-check
    failure on a POST_BEST_EFFORT / BEST_EFFORT_LOW_N subtype (catch
    + log + skip is acceptable, the trigger detection function does
    this and surfaces the count in cell 19) or (b) a registry-coverage
    bug (UNKNOWN_PLAYTYPE / FAIL / FAIL_NO_SAMPLES, which should never
    fire because cell 14 gates these but defense in depth raises here
    too)."""

    def __init__(self, message: str, category: str):
        super().__init__(message)
        self.category = category


# Cross-check tolerances for the two non-VERIFIED tiers (cell 14 verdicts).
# At scorer time we read the team's score direct, then look at the next
# clean play's score for the same team. If the absolute delta exceeds
# the tier's tolerance we raise (instead of silently using the direct
# read).
KO_RETURN_TD_INVARIANT_TOLERANCE = 2   # POST_BEST_EFFORT
LOW_N_INVARIANT_TOLERANCE = 7          # BEST_EFFORT_LOW_N

# Telemetry counters surfaced in cell 19's run summary.
KO_RETURN_TD_VERIFIED_BY_INVARIANT = 0
KO_RETURN_TD_RAISED_BY_INVARIANT = 0
LOW_N_VERIFIED_BY_INVARIANT = 0
LOW_N_RAISED_BY_INVARIANT = 0
SCORING_AMBIGUITY_LOG: list[dict] = []


def _post_play_scores_for_team(plays_sorted: list[dict], idx: int,
                                team: str) -> int | None:
    """Return the score for `team` AT THE END of plays_sorted[idx].

    Branching is by registry verdict (cell 14):

      * Non-scoring or PAT-like play -> direct read.
      * Scoring play, verdict POST_PLAY -> direct read (CFBD's
        offenseScore/defenseScore already reflect the post-play
        scoreboard for these subtypes).
      * Scoring play, verdict POST_BEST_EFFORT -> direct read with
        a ±2 cross-check against the next clean play's score for the
        same team. Raises ScoringPlayAmbiguityError(category=
        "CROSS_CHECK") on failure.
      * Scoring play, verdict BEST_EFFORT_LOW_N -> same as
        POST_BEST_EFFORT but with a ±7 tolerance.
      * Scoring play with playType not in registry, or verdict
        FAIL / FAIL_NO_SAMPLES -> raises immediately. Cell 14 gates
        these so they should never reach here in production.

    Returns the integer score, or None if the team is not on either
    side of the play (neutral-site oddities) or the score field is
    missing.
    """
    global KO_RETURN_TD_VERIFIED_BY_INVARIANT, KO_RETURN_TD_RAISED_BY_INVARIANT
    global LOW_N_VERIFIED_BY_INVARIANT, LOW_N_RAISED_BY_INVARIANT

    p = plays_sorted[idx]

    def _score_for_team(play: dict, team: str) -> int | None:
        if play.get("offense") == team:
            v = play.get("offenseScore")
            return int(v) if v is not None else None
        if play.get("defense") == team:
            v = play.get("defenseScore")
            return int(v) if v is not None else None
        return None

    direct = _score_for_team(p, team)

    if not bool(p.get("scoring")):
        return direct
    if _is_pat_like(p):
        return direct

    pt_l = (p.get("playType") or "").strip().lower()
    entry = SCORING_PLAY_REGISTRY.get(pt_l) if pt_l else None
    if entry is None:
        raise ScoringPlayAmbiguityError(
            f"play_id={p.get('id')} game_id={p.get('gameId')}: playType="
            f"{p.get('playType')!r} (scoring=true) is not in "
            f"SCORING_PLAY_REGISTRY (curated list KNOWN_SCORING_PLAYTYPES "
            f"is incomplete or playType is empty).",
            category="UNKNOWN_PLAYTYPE",
        )

    v = entry["verdict"]
    if v in ("FAIL", "FAIL_NO_SAMPLES"):
        raise ScoringPlayAmbiguityError(
            f"play_id={p.get('id')} game_id={p.get('gameId')}: playType="
            f"{p.get('playType')!r} verdict={v} post_share="
            f"{entry['post_share']}",
            category="FAIL",
        )
    if v == "POST_PLAY":
        return direct
    if v in ("POST_BEST_EFFORT", "BEST_EFFORT_LOW_N"):
        # Cross-check: read the next clean play's score for the same
        # team and compare to the direct read.
        nxt = _find_clean_next(plays_sorted, idx, team)
        tol = (KO_RETURN_TD_INVARIANT_TOLERANCE if v == "POST_BEST_EFFORT"
               else LOW_N_INVARIANT_TOLERANCE)
        if nxt is None or direct is None:
            return direct
        ns_raw = (nxt.get("offenseScore") if nxt.get("offense") == team
                  else nxt.get("defenseScore") if nxt.get("defense") == team
                  else None)
        if ns_raw is None:
            return direct
        delta = abs(int(ns_raw) - int(direct))
        if delta > tol:
            if v == "POST_BEST_EFFORT":
                KO_RETURN_TD_RAISED_BY_INVARIANT += 1
            else:
                LOW_N_RAISED_BY_INVARIANT += 1
            raise ScoringPlayAmbiguityError(
                f"play_id={p.get('id')} game_id={p.get('gameId')}: "
                f"verdict={v} cross-check delta={delta} > tolerance={tol}",
                category="CROSS_CHECK",
            )
        if v == "POST_BEST_EFFORT":
            KO_RETURN_TD_VERIFIED_BY_INVARIANT += 1
        else:
            LOW_N_VERIFIED_BY_INVARIANT += 1
        return direct

    raise ScoringPlayAmbiguityError(
        f"play_id={p.get('id')} game_id={p.get('gameId')}: unexpected "
        f"verdict={v!r} for playType={p.get('playType')!r}",
        category="UNKNOWN_VERDICT",
    )


def _clock_to_int(value: Any) -> int:
    """Coerce CFBD clock components (sometimes int, sometimes str) to int.

    Returns 0 for None / missing / empty string -- those are the only
    benign inputs we expect from `clock.minutes` / `clock.seconds`.
    Any other un-parseable string raises ValueError. Previously this
    silently returned 0, which masked bad clock data."""
    if value is None:
        return 0
    if isinstance(value, (int, np.integer)):
        return int(value)
    s = str(value).strip()
    if s == "":
        return 0
    return int(s)  # raises ValueError on any non-integer string


def detect_triggers_for_game(
    plays: list[dict],
    pregame: dict,
    drives: list[dict] | None = None,
) -> list[dict]:
    """Return list of trigger row dicts (one per crossed D, regulation only).

    `pregame` is the pregame_df row as a dict (provides fav_team, dog_team,
    pregame_spread, etc.). `drives` is the list of drives for this game (used
    to look up startTime / endTime when needed; currently driveId/driveNumber
    come straight from the play).
    """
    fav_team = pregame["fav_team"]
    dog_team = pregame["dog_team"]
    game_id = pregame["game_id"]

    # Canonical sort + dedup. Same (period, driveNumber, playNumber) helper
    # used by cell 14 (verification) and cell 17 (assert_no_lookahead) so
    # all three see the same play index for any given game.
    sorted_plays, _discards = _sorted_plays_for_game(plays)
    if not sorted_plays:
        return []

    triggers_seen: set[int] = set()
    rows: list[dict] = []

    for idx, p in enumerate(sorted_plays):
        period = int(p["period"])
        # Exclude OT plays entirely (A.2 / R15).
        if period >= 5:
            continue

        try:
            fav_score = _post_play_scores_for_team(sorted_plays, idx, fav_team)
            dog_score = _post_play_scores_for_team(sorted_plays, idx, dog_team)
        except ScoringPlayAmbiguityError as e:
            # Log + skip this play. Cell 19 surfaces the log and hard-fails
            # if any HARD category (UNKNOWN_PLAYTYPE / FAIL / FAIL_NO_SAMPLES /
            # UNKNOWN_VERDICT) shows up; CROSS_CHECK is the documented
            # acceptable case for the BEST_EFFORT tiers.
            SCORING_AMBIGUITY_LOG.append({
                "game_id": int(game_id),
                "play_number": int(p.get("playNumber") or 0),
                "play_id": p.get("id"),
                "play_type": p.get("playType"),
                "category": e.category,
                "message": str(e),
            })
            continue
        if fav_score is None or dog_score is None:
            continue

        deficit = int(dog_score) - int(fav_score)
        if deficit <= 0:
            continue

        for D in DEFICIT_THRESHOLDS:
            if D in triggers_seen:
                continue
            if deficit < D:
                continue
            triggers_seen.add(D)

            # --- Build the row ---
            clock = p.get("clock") or {}
            cm = _clock_to_int(clock.get("minutes"))
            cs = _clock_to_int(clock.get("seconds"))
            clock_secs_in_period = cm * 60 + cs
            secs_remaining_in_regulation = (4 - period) * 15 * 60 + clock_secs_in_period
            minutes_elapsed_in_period = (15 * 60 - clock_secs_in_period) / 60.0
            minutes_elapsed_total = (period - 1) * 15 + minutes_elapsed_in_period

            total_points_at_trigger = int(fav_score) + int(dog_score)
            points_per_minute = (
                round(total_points_at_trigger / minutes_elapsed_total, 4)
                if minutes_elapsed_total > 0 else None
            )

            possession_team = p.get("offense")
            fav_has_ball = (possession_team == fav_team)
            yardline_at_trigger = p.get("yardsToGoal")  # 0-100, offense-perspective
            distance_to_first_down = p.get("distance")
            down = p.get("down")

            rows.append({
                "game_id": int(game_id),
                "fav_deficit": int(D),
                "play_number": int(p["playNumber"]),
                "trigger_play_id": int(p["id"]) if p.get("id") is not None else None,
                "play_type": p.get("playType"),
                "quarter": int(period),
                "clock_minutes_remaining": int(cm),
                "clock_seconds_remaining": int(cs),
                "clock_seconds_in_period_total": int(clock_secs_in_period),
                "seconds_remaining_in_regulation": int(secs_remaining_in_regulation),
                "minutes_elapsed_total": round(minutes_elapsed_total, 4),
                "fav_score_at_trigger": int(fav_score),
                "dog_score_at_trigger": int(dog_score),
                "actual_deficit_at_trigger": int(deficit),
                "total_points_at_trigger": int(total_points_at_trigger),
                "points_per_minute": points_per_minute,
                "possession_team": possession_team,
                "fav_has_ball": bool(fav_has_ball),
                "yardline_at_trigger": int(yardline_at_trigger) if yardline_at_trigger is not None else None,
                "distance_to_first_down": int(distance_to_first_down) if distance_to_first_down is not None else None,
                "down": int(down) if down is not None else None,
                "trigger_drive_id": int(p["driveId"]) if p.get("driveId") is not None else None,
                "drive_number_in_game": int(p["driveNumber"]) if p.get("driveNumber") is not None else None,
            })

    # Sort and assign trigger_sequence (ordinal within game). play_number is
    # per-drive; we add drive_number_in_game and quarter to break ties so
    # the ordinal reflects chronological order even if a game has same-
    # play_number rows on different drives.
    rows.sort(key=lambda r: (r["quarter"], r["drive_number_in_game"] or 0,
                             r["play_number"], r["fav_deficit"]))
    for k, r in enumerate(rows, start=1):
        r["trigger_sequence"] = k

    return rows


print("[ok] detect_triggers_for_game defined")

In [ ]:
# --- assert_no_lookahead: per-row hard gate (R3) -------------------------
# This is the last line of defense. Every emitted row passes through here
# before being added to the final DataFrame. If a row depends on any play
# downstream of the trigger play in the canonical chronological order
# (period, driveNumber, playNumber), or if score re-derivation disagrees
# with what trigger detection emitted, the assertion fires and we stop
# the whole notebook -- no silent lookahead bias.
class LookaheadError(AssertionError):
    """Raised when a trigger_events row references future play data."""


def assert_no_lookahead(row: dict, plays_for_game: list[dict],
                          pregame: dict) -> None:
    """Verify the row's point-in-time fields are derivable from plays at
    or before the trigger play in chronological order. Raises
    LookaheadError on violation.

    Pre-game context fields are exempt (observable before kickoff).
    Label/outcome fields live in trigger_outcomes.csv (sibling artifact),
    not in the feature-safe trigger_events.csv (R3 structural split).
    """
    def _norm_play_id(pid: Any) -> int | None:
        if pid is None:
            return None
        return int(pid)

    trigger_pn = int(row["play_number"])
    row_tid = _norm_play_id(row.get("trigger_play_id"))

    # Same canonical sort + dedup helper as detect_triggers_for_game uses,
    # so re-derivation runs against the identical play index.
    sorted_plays, _discards = _sorted_plays_for_game(plays_for_game)
    if not sorted_plays:
        raise LookaheadError(
            f"game_id={row['game_id']} fav_deficit={row['fav_deficit']}: "
            f"no plays with (period, driveNumber, playNumber) populated"
        )

    # Resolve the exact trigger play by stable CFBD id. playNumber alone is
    # per-drive (collides across drives in the same period) so an id-first
    # lookup is mandatory. We fall back to (driveNumber, playNumber) if
    # the row carries no id.
    trigger_play: dict | None = None
    if row_tid is not None:
        for p in sorted_plays:
            if _norm_play_id(p.get("id")) == row_tid:
                trigger_play = p
                break
    if trigger_play is None:
        row_drv = row.get("drive_number_in_game")
        if row_drv is not None:
            trigger_play = next(
                (p for p in sorted_plays
                 if int(p["driveNumber"]) == int(row_drv)
                 and int(p["playNumber"]) == trigger_pn),
                None,
            )
    if trigger_play is None:
        raise LookaheadError(
            f"game_id={row['game_id']} fav_deficit={row['fav_deficit']}: "
            f"trigger play not found (play_number={trigger_pn}, "
            f"drive_number={row.get('drive_number_in_game')!r}, "
            f"trigger_play_id={row_tid!r})"
        )

    if int(trigger_play["playNumber"]) != trigger_pn:
        raise LookaheadError(
            f"game_id={row['game_id']}: play_number {trigger_pn} does not "
            f"match the play for trigger_play_id={row_tid} (play has "
            f"playNumber={trigger_play.get('playNumber')!r})"
        )

    # 1) trigger_play_id matches the resolved play (redundant when lookup
    #    was by id, but a hard guarantee for the (driveNumber, playNumber)
    #    fallback path).
    if row_tid is not None and row_tid != _norm_play_id(trigger_play.get("id")):
        raise LookaheadError(
            f"game_id={row['game_id']}: trigger_play_id mismatch "
            f"(row={row['trigger_play_id']!r}, trigger play="
            f"{trigger_play.get('id')!r})"
        )

    # 2) Score fields -- re-derive using the same sorted index that trigger
    #    detection used. Any disagreement is either a lookahead leak or a
    #    sort/dedup bug; either way we stop.
    re_idx = next(i for i, p in enumerate(sorted_plays) if p is trigger_play)
    try:
        re_fav = _post_play_scores_for_team(sorted_plays, re_idx, pregame["fav_team"])
        re_dog = _post_play_scores_for_team(sorted_plays, re_idx, pregame["dog_team"])
    except ScoringPlayAmbiguityError as e:
        raise LookaheadError(
            f"game_id={row['game_id']}: scorer raised "
            f"{e.category} during re-derivation -- emitted row "
            f"{(row['game_id'], row['fav_deficit'])} but the same scorer "
            f"call now refuses; sort/dedup or registry bug. {e}"
        )
    if re_fav != row["fav_score_at_trigger"]:
        raise LookaheadError(
            f"game_id={row['game_id']}: re-derived fav score "
            f"{re_fav} != row {row['fav_score_at_trigger']}"
        )
    if re_dog != row["dog_score_at_trigger"]:
        raise LookaheadError(
            f"game_id={row['game_id']}: re-derived dog score "
            f"{re_dog} != row {row['dog_score_at_trigger']}"
        )

    # 3) Quarter must be <= 4 (regulation only -- A.2 / R15).
    if int(row["quarter"]) >= 5:
        raise LookaheadError(
            f"game_id={row['game_id']}: OT trigger leaked into output "
            f"(quarter={row['quarter']})"
        )


print("[ok] assert_no_lookahead defined")

## Phase 0d — Build trigger_events DataFrame

Walk the working set, run `detect_triggers_for_game` on each, gate every emitted row through `assert_no_lookahead`, then merge with `pregame_df` to attach pre-game context (favorite, spread, ratings, ML, opening/closing/movement).

In [ ]:
# --- Iterate working-set games and build raw trigger rows --------------
all_trigger_rows: list[dict] = []
games_without_plays: list[int] = []
games_no_trigger: list[int] = []  # games where favorite never trailed
games_with_at_least_one: int = 0

t_start = time.perf_counter()
for gid, pregame_row in pregame_df.iterrows():
    pregame_dict = pregame_row.to_dict()
    pregame_dict["game_id"] = int(gid)
    plays = plays_by_game.get(int(gid))
    if not plays:
        games_without_plays.append(int(gid))
        continue
    drives = drives_by_game.get(int(gid))
    rows = detect_triggers_for_game(plays, pregame_dict, drives)
    if not rows:
        games_no_trigger.append(int(gid))
        continue
    games_with_at_least_one += 1
    # Per-row hard gate.
    for r in rows:
        assert_no_lookahead(r, plays, pregame_dict)
    all_trigger_rows.extend(rows)

elapsed = time.perf_counter() - t_start
print(f"trigger detection complete in {elapsed:.1f}s")
print(f"  working-set games processed: {len(pregame_df)}")
print(f"  games with trigger rows:     {games_with_at_least_one}")
print(f"  games where fav never trailed in regulation: {len(games_no_trigger)}")
print(f"  games missing play data (skipped): {len(games_without_plays)}")
print(f"  total trigger rows emitted: {len(all_trigger_rows)}")
print(f"\n[ok] all rows passed assert_no_lookahead")

# --- Score-state cross-check + sort-dedup telemetry ----------------------
print(f"\nScore-state cross-check telemetry:")
print(f"  POST_BEST_EFFORT (KO Return TD) +/-{KO_RETURN_TD_INVARIANT_TOLERANCE} verified: {KO_RETURN_TD_VERIFIED_BY_INVARIANT:,}")
print(f"  POST_BEST_EFFORT (KO Return TD) +/-{KO_RETURN_TD_INVARIANT_TOLERANCE} raised:   {KO_RETURN_TD_RAISED_BY_INVARIANT:,}")
print(f"  BEST_EFFORT_LOW_N               +/-{LOW_N_INVARIANT_TOLERANCE} verified: {LOW_N_VERIFIED_BY_INVARIANT:,}")
print(f"  BEST_EFFORT_LOW_N               +/-{LOW_N_INVARIANT_TOLERANCE} raised:   {LOW_N_RAISED_BY_INVARIANT:,}")
print(f"\nSORT_DEDUP_LOG: {len(SORT_DEDUP_LOG):,} games with duplicate "
      f"(period, driveNumber, playNumber) rows "
      f"({sum(e['discards'] for e in SORT_DEDUP_LOG):,} rows discarded total)")

# --- Ambiguity log: hard-fail if any non-CROSS_CHECK category appeared ---
from collections import Counter as _Counter
if SCORING_AMBIGUITY_LOG:
    by_cat = _Counter(e["category"] for e in SCORING_AMBIGUITY_LOG)
    print(f"\nSCORING_AMBIGUITY_LOG: {len(SCORING_AMBIGUITY_LOG):,} events")
    for cat, n in by_cat.most_common():
        print(f"  {cat}: {n:,}")
    _hard = {"UNKNOWN_PLAYTYPE", "FAIL", "FAIL_NO_SAMPLES", "UNKNOWN_VERDICT"}
    _hard_events = [e for e in SCORING_AMBIGUITY_LOG if e["category"] in _hard]
    if _hard_events:
        # Cell 14 should have gated these. Their appearance here is a
        # registry/data bug we want to surface, not silently swallow.
        for e in _hard_events[:5]:
            print(f"  example: {e}")
        raise AssertionError(
            f"{len(_hard_events)} hard-category ambiguity events at scorer "
            f"time despite cell-14 gating. Investigate before proceeding."
        )
else:
    print(f"\nSCORING_AMBIGUITY_LOG: 0 events")

In [ ]:
# --- Merge with pregame_df to attach pre-game context fields -----------
triggers_df = pd.DataFrame(all_trigger_rows)

# Pre-game columns only — LABEL / outcome fields go to trigger_outcomes.csv
# (R3 structural split) so feature notebooks never parse them from the same
# file as in-game state.
pregame_cols_to_merge = [
    "game_id",
    "season", "season_type", "week",
    "fav_team", "dog_team", "home_team", "away_team", "home_is_fav",
    "pregame_spread", "pregame_spread_provider",
    "pregame_fav_ml", "pregame_dog_ml", "pregame_ml_provider",
    "fav_pregame_rating", "dog_pregame_rating", "rating_gap",
    "opening_spread", "opening_spread_provider",
    "closing_spread", "spread_movement", "line_moved_against_fav",
]

triggers_df = triggers_df.merge(
    pregame_df.reset_index()[pregame_cols_to_merge],
    on="game_id",
    how="left",
    validate="many_to_one",
)

# Final column order (logical grouping for human readability + downstream
# consumers). Schema sidecar documents semantics.
final_columns = [
    # IDs & sequencing
    "game_id", "fav_deficit", "trigger_sequence",
    # Game metadata
    "season", "season_type", "week",
    # Teams
    "fav_team", "dog_team", "home_team", "away_team", "home_is_fav",
    # Pre-game market
    "pregame_spread", "pregame_spread_provider",
    "pregame_fav_ml", "pregame_dog_ml", "pregame_ml_provider",
    "opening_spread", "opening_spread_provider",
    "closing_spread", "spread_movement", "line_moved_against_fav",
    # Pre-game ratings (A.7)
    "fav_pregame_rating", "dog_pregame_rating", "rating_gap",
    # Trigger play identity (V5 names trigger_play_id / trigger_drive_id)
    "trigger_play_id", "play_number", "play_type",
    # Trigger play timing
    "quarter", "clock_minutes_remaining", "clock_seconds_remaining",
    "clock_seconds_in_period_total", "seconds_remaining_in_regulation",
    "minutes_elapsed_total",
    # Trigger play game state
    "fav_score_at_trigger", "dog_score_at_trigger",
    "actual_deficit_at_trigger", "total_points_at_trigger", "points_per_minute",
    "possession_team", "fav_has_ball",
    "yardline_at_trigger", "distance_to_first_down", "down",
    "trigger_drive_id", "drive_number_in_game",
]
missing = set(final_columns) - set(triggers_df.columns)
extra = set(triggers_df.columns) - set(final_columns)
assert not missing, f"missing columns: {missing}"
assert not extra, f"extra columns not in final_columns: {extra}"
triggers_df = triggers_df[final_columns].copy()

# --- Sibling outcomes table: V5 DDL OUTCOME block, same row order --------
_outcome_cols = ["final_fav_won", "final_fav_score", "final_dog_score", "margin_of_defeat"]
outcomes_df = triggers_df[["game_id", "fav_deficit"]].merge(
    pregame_df.reset_index()[["game_id", *_outcome_cols]].drop_duplicates(subset=["game_id"]),
    on="game_id",
    how="left",
    validate="many_to_one",
)
assert len(outcomes_df) == len(triggers_df), "outcomes row count != trigger_events"
_oc_order = ["game_id", "fav_deficit", *_outcome_cols]
assert set(outcomes_df.columns) == set(_oc_order)
outcomes_df = outcomes_df[_oc_order].copy()

print(f"trigger_events shape: {triggers_df.shape}")
print(f"trigger_outcomes shape: {outcomes_df.shape}")
print(f"\ntrigger_events columns ({len(triggers_df.columns)}):")
for _c in triggers_df.columns:
    print("  " + str(_c))


In [ ]:
# --- Schema validation + UNIQUE constraint check ----------------------
# 1) UNIQUE(game_id, fav_deficit) per BUILD_SPEC.md
dup_mask = triggers_df.duplicated(subset=["game_id", "fav_deficit"])
assert not dup_mask.any(), (
    f"UNIQUE(game_id, fav_deficit) violated by {int(dup_mask.sum())} rows"
)

# 2) fav_deficit values must be in DEFICIT_THRESHOLDS
bad_d = ~triggers_df["fav_deficit"].isin(DEFICIT_THRESHOLDS)
assert not bad_d.any(), (
    f"fav_deficit out of allowed set: {triggers_df.loc[bad_d, 'fav_deficit'].unique()}"
)

# 3) actual_deficit_at_trigger >= fav_deficit (the trigger fired BECAUSE deficit reached threshold)
bad_def = triggers_df["actual_deficit_at_trigger"] < triggers_df["fav_deficit"]
assert not bad_def.any(), (
    f"{int(bad_def.sum())} rows have actual deficit < threshold (broken detection)"
)

# 4) quarter <= 4 (regulation only)
bad_q = triggers_df["quarter"] >= 5
assert not bad_q.any(), (
    f"{int(bad_q.sum())} OT rows leaked into output"
)

# 5) pregame_spread always <= 0
bad_spread = triggers_df["pregame_spread"] > 0
assert not bad_spread.any(), (
    f"{int(bad_spread.sum())} rows have positive pregame_spread (sign convention violated)"
)

# 6) trigger_sequence is positive and forms a contiguous 1..N per game
seq_check = triggers_df.groupby("game_id")["trigger_sequence"].agg(["min", "max", "count"])
bad_seq = (seq_check["min"] != 1) | (seq_check["max"] != seq_check["count"])
assert not bad_seq.any(), (
    f"{int(bad_seq.sum())} games have non-contiguous trigger_sequence"
)

# 7) spread_movement: NULL iff opening_spread is NULL (R16)
mismatch = (
    triggers_df["spread_movement"].isna() != triggers_df["opening_spread"].isna()
)
assert not mismatch.any(), (
    f"{int(mismatch.sum())} rows have spread_movement/opening_spread NULL mismatch"
)

# 8) clock_minutes_remaining in [0, 14], clock_seconds_remaining in [0, 59]
bad_cm = (triggers_df["clock_minutes_remaining"] < 0) | (triggers_df["clock_minutes_remaining"] > 15)
bad_cs = (triggers_df["clock_seconds_remaining"] < 0) | (triggers_df["clock_seconds_remaining"] > 59)
assert not bad_cm.any(), f"{int(bad_cm.sum())} rows have invalid clock minutes"
assert not bad_cs.any(), f"{int(bad_cs.sum())} rows have invalid clock seconds"

# 9) seconds_remaining_in_regulation in [0, 3600]
bad_sr = (triggers_df["seconds_remaining_in_regulation"] < 0) | (triggers_df["seconds_remaining_in_regulation"] > 3600)
assert not bad_sr.any(), f"{int(bad_sr.sum())} rows have invalid seconds_remaining_in_regulation"

# 10) yardline_at_trigger in [0, 100] (when not NULL)
yl_nn = triggers_df["yardline_at_trigger"].dropna()
bad_yl = (yl_nn < 0) | (yl_nn > 100)
assert not bad_yl.any(), f"{int(bad_yl.sum())} rows have invalid yardline_at_trigger"

# 11) trigger_outcomes: OUTCOME / LABEL field consistency
fav_lost_mask = outcomes_df["final_fav_won"] == False  # noqa: E712
fav_won_mask = outcomes_df["final_fav_won"] == True   # noqa: E712
mod_when_lost_null = fav_lost_mask & outcomes_df["margin_of_defeat"].isna()
assert not mod_when_lost_null.any(), (
    f"{int(mod_when_lost_null.sum())} outcome rows: fav_won=False but margin_of_defeat is NULL"
)
mod_when_lost_neg = fav_lost_mask & (outcomes_df["margin_of_defeat"] <= 0)
assert not mod_when_lost_neg.any(), (
    f"{int(mod_when_lost_neg.sum())} outcome rows: fav_won=False but margin_of_defeat <= 0"
)
mod_when_won_set = fav_won_mask & outcomes_df["margin_of_defeat"].notna()
assert not mod_when_won_set.any(), (
    f"{int(mod_when_won_set.sum())} outcome rows: fav_won=True but margin_of_defeat is set"
)
score_mismatch_lost = fav_lost_mask & (
    outcomes_df["final_fav_score"] >= outcomes_df["final_dog_score"]
)
assert not score_mismatch_lost.any(), (
    f"{int(score_mismatch_lost.sum())} outcome rows: fav_won=False but final_fav_score >= final_dog_score"
)
score_mismatch_won = fav_won_mask & (
    outcomes_df["final_fav_score"] <= outcomes_df["final_dog_score"]
)
assert not score_mismatch_won.any(), (
    f"{int(score_mismatch_won.sum())} outcome rows: fav_won=True but final_fav_score <= final_dog_score"
)

print(f"[ok] trigger_events: {10} schema constraints passed on {len(triggers_df):,} rows")
print(f"[ok] trigger_outcomes: outcome field consistency on {len(outcomes_df):,} rows")
print(f"\nrows by fav_deficit:")
print(triggers_df["fav_deficit"].value_counts().sort_index().to_string())
print(f"\nrows by quarter:")
print(triggers_df["quarter"].value_counts().sort_index().to_string())
print(f"\nrows by season:")
print(triggers_df.groupby(["season", "season_type"]).size().to_string())

In [ ]:
# --- 10% sample re-detection spot-check (R3 belt-and-suspenders) -----
# Take a random 10% sample of trigger rows (seeded), and for each row
# re-run trigger detection from scratch using a freshly-loaded plays
# subset, then compare. Any discrepancy fires.
rng = np.random.default_rng(seed=42)
sample_size = max(1, int(len(triggers_df) * 0.10))
sample_idx = rng.choice(len(triggers_df), size=sample_size, replace=False)
sample_df = triggers_df.iloc[sample_idx].copy()
print(f"spot-check sample: {len(sample_df):,} rows ({sample_size / len(triggers_df) * 100:.1f}% of total)")

# Group by game so we re-detect once per game (more efficient than once per row).
sample_games = sorted(sample_df["game_id"].unique())
discrepancies: list[dict] = []
for gid in sample_games:
    pregame_dict = pregame_df.loc[gid].to_dict()
    pregame_dict["game_id"] = int(gid)
    plays = plays_by_game.get(int(gid))
    if not plays:
        continue
    drives = drives_by_game.get(int(gid))
    re_rows = detect_triggers_for_game(plays, pregame_dict, drives)
    re_df = pd.DataFrame(re_rows)
    if re_df.empty:
        continue
    # Compare each sampled row's deficits to re-detection.
    for _, srow in sample_df[sample_df["game_id"] == gid].iterrows():
        D = int(srow["fav_deficit"])
        match = re_df[re_df["fav_deficit"] == D]
        if match.empty:
            discrepancies.append({"game_id": gid, "D": D, "issue": "re-detect missing this D"})
            continue
        m = match.iloc[0]
        for col in ("play_number", "fav_score_at_trigger", "dog_score_at_trigger",
                    "actual_deficit_at_trigger", "quarter"):
            if int(m[col]) != int(srow[col]):
                discrepancies.append({
                    "game_id": gid, "D": D, "col": col,
                    "original": int(srow[col]), "redetect": int(m[col]),
                })

if discrepancies:
    print(f"[!!] {len(discrepancies)} discrepancies found in spot-check:")
    for d in discrepancies[:20]:
        print(f"  {d}")
    raise RuntimeError("spot-check failed — trigger detection is non-deterministic")
print(f"[ok] spot-check passed: re-detection matches original for all {len(sample_df):,} rows "
      f"across {len(sample_games):,} games")

## Phase 0e — Outputs

Write five files to `research/results/`: the feature-safe `trigger_events` CSV, the sibling `trigger_outcomes` LABEL CSV, bucket counts, and two schema sidecars (`trigger_events.schema.md` + `trigger_outcomes.schema.md`).

In [ ]:
# --- Write trigger_events.csv + trigger_outcomes.csv -------------------
TRIGGER_EVENTS_CSV = RESULTS_DIR / "trigger_events.csv"
TRIGGER_OUTCOMES_CSV = RESULTS_DIR / "trigger_outcomes.csv"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Stable sort for reproducibility — apply identical row order to sibling outcomes.
_sort_idx = triggers_df.sort_values(
    ["season", "season_type", "week", "game_id", "fav_deficit"]
).index
triggers_df_sorted = triggers_df.loc[_sort_idx].reset_index(drop=True)
outcomes_df_sorted = outcomes_df.loc[_sort_idx].reset_index(drop=True)

triggers_df_sorted.to_csv(TRIGGER_EVENTS_CSV, index=False)
size_kb = TRIGGER_EVENTS_CSV.stat().st_size / 1024
print(f"[ok] wrote {TRIGGER_EVENTS_CSV.relative_to(REPO_ROOT)}: "
      f"{len(triggers_df_sorted):,} rows × {len(triggers_df_sorted.columns)} cols "
      f"({size_kb:,.1f} KB)")

outcomes_df_sorted.to_csv(TRIGGER_OUTCOMES_CSV, index=False)
size_kb_o = TRIGGER_OUTCOMES_CSV.stat().st_size / 1024
print(f"[ok] wrote {TRIGGER_OUTCOMES_CSV.relative_to(REPO_ROOT)}: "
      f"{len(outcomes_df_sorted):,} rows × {len(outcomes_df_sorted.columns)} cols "
      f"({size_kb_o:,.1f} KB)")

In [ ]:
# --- Write trigger_events_bucket_counts.csv -----------------------
# Per (season, season_type, fav_deficit, quarter). This anticipates the
# Phase 0 acceptance gate 4 thinness-check used in N03's walk-forward
# splits — it lets a future reader see at a glance whether any
# (deficit, quarter) bucket is degenerate within a season.
bucket_counts_df = (
    triggers_df_sorted
    .groupby(["season", "season_type", "fav_deficit", "quarter"])
    .size()
    .reset_index(name="n_triggers")
    .sort_values(["season", "season_type", "fav_deficit", "quarter"])
)
BUCKETS_CSV = RESULTS_DIR / "trigger_events_bucket_counts.csv"
bucket_counts_df.to_csv(BUCKETS_CSV, index=False)
size_kb = BUCKETS_CSV.stat().st_size / 1024
print(f"[ok] wrote {BUCKETS_CSV.relative_to(REPO_ROOT)}: "
      f"{len(bucket_counts_df):,} rows ({size_kb:,.1f} KB)")
print(f"\nbucket counts (first 30 rows):")
print(bucket_counts_df.head(30).to_string(index=False))

In [ ]:
# --- Write trigger_events.schema.md (sidecar) -----------------------
# Every column documented with type, unit, sign convention, lookahead
# category. Provider priority. Empirical pre/post-play findings.
# Generation date + source commit hash.
SCHEMA_MD = RESULTS_DIR / "trigger_events.schema.md"
OUTCOMES_SCHEMA_MD = RESULTS_DIR / "trigger_outcomes.schema.md"

# Source commit hash (the commit containing this notebook's source).
try:
    source_commit = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=str(REPO_ROOT),
        capture_output=True, text=True, check=True
    ).stdout.strip()
except Exception:
    source_commit = "unknown (git unavailable)"

# Column metadata: (name, type, unit, sign/range, category, description)
# Categories:
#   IDENTIFIER               — primary key components, internal counters
#   PRE-GAME CONTEXT         — observable BEFORE kickoff (R3-safe by definition)
#   POINT-IN-TIME GAME STATE — derivable from plays with playNumber <= trigger.play_number only (R3-safe by construction; verified by assert_no_lookahead)
COLUMN_DOCS: list[tuple[str, str, str, str, str, str]] = [
    # IDENTIFIER
    ("game_id", "INTEGER", "—", "CFBD game ID", "IDENTIFIER",
     "Primary key component. Joins to trigger_features.game_id and to the games metadata."),
    ("fav_deficit", "INTEGER", "points", "{3, 7, 10, 14, 21}", "IDENTIFIER",
     "Primary key component. Threshold value, NOT the actual deficit observed (see actual_deficit_at_trigger)."),
    ("trigger_sequence", "INTEGER", "—", "1..N within game", "IDENTIFIER",
     "Ordinal index of this trigger within its game, sorted by (play_number ASC, fav_deficit ASC). seq=1 is the first crossed threshold."),

    # PRE-GAME CONTEXT
    ("season", "INTEGER", "year", "2015..2024", "PRE-GAME CONTEXT",
     "Calendar year the season started (e.g., 2024 for the 2024-25 season)."),
    ("season_type", "TEXT", "—", "regular | postseason", "PRE-GAME CONTEXT",
     "Regular season or bowl/playoff. Excludes any 'preseason' or 'spring' values (none in our corpus)."),
    ("week", "INTEGER", "—", "1..16 (regular), 1 (postseason)", "PRE-GAME CONTEXT",
     "Week within season. Postseason games are typically week=1."),
    ("fav_team", "TEXT", "—", "school name", "PRE-GAME CONTEXT",
     "Pre-game favorite per pregame_spread sign. Pick'em games (spread==0) are excluded entirely."),
    ("dog_team", "TEXT", "—", "school name", "PRE-GAME CONTEXT",
     "Pre-game underdog per pregame_spread sign."),
    ("home_team", "TEXT", "—", "school name", "PRE-GAME CONTEXT",
     "Home team (geographic/scheduling, not favorite-related)."),
    ("away_team", "TEXT", "—", "school name", "PRE-GAME CONTEXT",
     "Away team."),
    ("home_is_fav", "BOOLEAN", "—", "True | False", "PRE-GAME CONTEXT",
     "True iff fav_team == home_team."),
    ("pregame_spread", "REAL", "points", "always <= 0 (favorite negative)", "PRE-GAME CONTEXT",
     "Closing spread from favorite's perspective. Sign convention: -7.0 means favorite is laying 7 points. Pick'em (0) excluded."),
    ("pregame_spread_provider", "TEXT", "—", "provider name", "PRE-GAME CONTEXT",
     "Sportsbook actually used for pregame_spread, per priority list (see Provider priority section below)."),
    ("pregame_fav_ml", "REAL", "American odds", "negative for favorite", "PRE-GAME CONTEXT",
     "Closing moneyline price on the favorite. NULL if no real book has ML for this game (frequent in 2018-2022)."),
    ("pregame_dog_ml", "REAL", "American odds", "positive for underdog", "PRE-GAME CONTEXT",
     "Closing moneyline price on the underdog. NULL with same coverage as pregame_fav_ml."),
    ("pregame_ml_provider", "TEXT", "—", "provider name | NULL", "PRE-GAME CONTEXT",
     "Sportsbook actually used for pregame_fav_ml/pregame_dog_ml. Real books only — consensus is excluded for ML."),
    ("opening_spread", "REAL", "points", "<= 0 (closing-fav perspective)", "PRE-GAME CONTEXT",
     "Opening spread expressed in closing favorite's perspective. NULL if no acceptable provider has spreadOpen for this game (~30% missing rate)."),
    ("opening_spread_provider", "TEXT", "—", "provider name | NULL", "PRE-GAME CONTEXT",
     "Sportsbook actually used for opening_spread."),
    ("closing_spread", "REAL", "points", "always <= 0 (favorite negative)", "PRE-GAME CONTEXT",
     "Same as pregame_spread (the closing spread IS the pre-game spread for purposes of trigger_events)."),
    ("spread_movement", "REAL", "points", "negative = fav got more favored", "PRE-GAME CONTEXT",
     "closing_spread - opening_spread (in closing-fav's perspective). NULL when opening_spread is NULL (R16). Negative = favorite getting more favored from open to close."),
    ("line_moved_against_fav", "BOOLEAN", "—", "True | False | NULL", "PRE-GAME CONTEXT",
     "True iff spread_movement > 0. NULL when spread_movement is NULL."),
    ("fav_pregame_rating", "REAL", "Elo points", "typically 1200..2100", "PRE-GAME CONTEXT",
     "Favorite's pre-game Elo (CFBD homePregameElo / awayPregameElo). A.7 substitution for SP+/FPI which leak."),
    ("dog_pregame_rating", "REAL", "Elo points", "typically 1200..2100", "PRE-GAME CONTEXT",
     "Underdog's pre-game Elo. NULL only if CFBD's Elo is missing for that team in that game (rare per N00 audit, all 20 (season, type) pairs cleared 80%)."),
    ("rating_gap", "REAL", "Elo points", "always >= 0 typically", "PRE-GAME CONTEXT",
     "fav_pregame_rating - dog_pregame_rating. May be slightly negative when home-field-advantage flipped the favorite vs the rating ranking; the favorite is determined by SPREAD, not Elo."),

    # POINT-IN-TIME GAME STATE
    ("trigger_play_id", "INTEGER", "—", "CFBD play ID", "POINT-IN-TIME GAME STATE",
     "V5 DDL name. Unique CFBD identifier for the play that caused this trigger — distinct from any hypothetical current/next play fields. Joins to /plays."),
    ("play_number", "INTEGER", "—", "1..~180 typical", "POINT-IN-TIME GAME STATE",
     "Sequential play number within the game (CFBD playNumber). The trigger is the FIRST play after which fav_deficit was reached."),
    ("play_type", "TEXT", "—", "CFBD playType string", "POINT-IN-TIME GAME STATE",
     "Type of play (e.g., 'Touchdown', 'Field Goal Good', 'Pass Reception', 'Interception Return Touchdown'). Free text per CFBD."),
    ("quarter", "INTEGER", "—", "1, 2, 3, 4 (regulation only)", "POINT-IN-TIME GAME STATE",
     "Quarter (CFBD period). OT (period >= 5) excluded entirely per A.2/R15."),
    ("clock_minutes_remaining", "INTEGER", "minutes", "0..15", "POINT-IN-TIME GAME STATE",
     "Minutes-portion of the game clock at the start of the trigger play."),
    ("clock_seconds_remaining", "INTEGER", "seconds", "0..59", "POINT-IN-TIME GAME STATE",
     "Seconds-portion of the game clock at the start of the trigger play."),
    ("clock_seconds_in_period_total", "INTEGER", "seconds", "0..900", "POINT-IN-TIME GAME STATE",
     "Total seconds remaining in the current quarter (clock_minutes * 60 + clock_seconds)."),
    ("seconds_remaining_in_regulation", "INTEGER", "seconds", "0..3600", "POINT-IN-TIME GAME STATE",
     "Total seconds remaining until the end of regulation (Q4 0:00)."),
    ("minutes_elapsed_total", "REAL", "minutes", "0.0..60.0", "POINT-IN-TIME GAME STATE",
     "Minutes of game elapsed at the START of the trigger play. (15 - clock) + (period-1)*15."),
    ("fav_score_at_trigger", "INTEGER", "points", ">=0", "POINT-IN-TIME GAME STATE",
     "Favorite's score at the END of the trigger play."),
    ("dog_score_at_trigger", "INTEGER", "points", ">=0", "POINT-IN-TIME GAME STATE",
     "Underdog's score at the END of the trigger play."),
    ("actual_deficit_at_trigger", "INTEGER", "points", ">=fav_deficit", "POINT-IN-TIME GAME STATE",
     "dog_score - fav_score at trigger. Exceeds fav_deficit when crossing happened on a multi-point play (e.g., a 7-pt TD on a 0-0 game crosses both D=3 and D=7 simultaneously)."),
    ("total_points_at_trigger", "INTEGER", "points", "fav + dog", "POINT-IN-TIME GAME STATE",
     "fav_score_at_trigger + dog_score_at_trigger."),
    ("points_per_minute", "REAL", "points/min", ">=0 | NULL", "POINT-IN-TIME GAME STATE",
     "total_points_at_trigger / minutes_elapsed_total. NULL when minutes_elapsed_total == 0."),
    ("possession_team", "TEXT", "—", "school name", "POINT-IN-TIME GAME STATE",
     "Team on offense at the trigger play (CFBD `offense`)."),
    ("fav_has_ball", "BOOLEAN", "—", "True | False", "POINT-IN-TIME GAME STATE",
     "True iff possession_team == fav_team."),
    ("yardline_at_trigger", "INTEGER", "yards", "0..100 (offense-perspective)", "POINT-IN-TIME GAME STATE",
     "CFBD `yardsToGoal`. Distance to defense's end zone from offense's perspective. 0=defense's goal line, 100=offense's own goal line."),
    ("distance_to_first_down", "INTEGER", "yards", ">=0 | NULL", "POINT-IN-TIME GAME STATE",
     "CFBD `distance`. Yards needed for first down. NULL on certain non-snap plays."),
    ("down", "INTEGER", "—", "1..4 | NULL", "POINT-IN-TIME GAME STATE",
     "Down number. NULL on kickoffs, PATs, certain 2pt conversions, etc."),
    ("trigger_drive_id", "INTEGER", "—", "CFBD drive ID | NULL", "POINT-IN-TIME GAME STATE",
     "V5 DDL name. Drive that contains the trigger play. NULL if CFBD didn't assign one."),
    ("drive_number_in_game", "INTEGER", "—", "1..N | NULL", "POINT-IN-TIME GAME STATE",
     "1-indexed drive ordinal in the game (CFBD `driveNumber`)."),
]

# Sanity: every actual column documented; no orphan docs.
documented = {c[0] for c in COLUMN_DOCS}
actual = set(triggers_df_sorted.columns)
assert documented == actual, (
    f"Schema doc mismatch.\n"
    f"  documented but not in CSV: {documented - actual}\n"
    f"  in CSV but not documented: {actual - documented}"
)

# Empirical findings -- one row per curated subtype, sorted by tier then
# alphabetically within tier so the higher-confidence rows appear first.
_TIER_ORDER = ("POST_PLAY", "POST_BEST_EFFORT", "BEST_EFFORT_LOW_N",
               "FAIL", "FAIL_NO_SAMPLES")


def _tier_rank(verdict: str) -> int:
    return _TIER_ORDER.index(verdict) if verdict in _TIER_ORDER else len(_TIER_ORDER)


score_state_table = (
    "| Play type | Verdict | Verified-POST / Verified-PRE / Total | POST share |\n"
    "|---|---|---|---:|\n"
)
for pt_l in sorted(SCORING_PLAY_REGISTRY.keys(),
                   key=lambda k: (_tier_rank(SCORING_PLAY_REGISTRY[k]["verdict"]), k)):
    r = SCORING_PLAY_REGISTRY[pt_l]
    ps_str = f"{100 * r['post_share']:.1f}%" if r["post_share"] is not None else "-"
    score_state_table += (
        f"| `{pt_l}` | `{r['verdict']}` | "
        f"{r['verified_post']} / {r['verified_pre']} / {r['n_samples']} | "
        f"{ps_str} |\n"
    )


def _tier_list(target: str) -> str:
    items = sorted(pt for pt, r in SCORING_PLAY_REGISTRY.items()
                   if r["verdict"] == target)
    if not items:
        return "(none)"
    return ", ".join(f"`{pt}`" for pt in items)


_n_post_tier = sum(1 for r in SCORING_PLAY_REGISTRY.values() if r["verdict"] == "POST_PLAY")
_n_be_tier   = sum(1 for r in SCORING_PLAY_REGISTRY.values() if r["verdict"] == "POST_BEST_EFFORT")
_n_low_tier  = sum(1 for r in SCORING_PLAY_REGISTRY.values() if r["verdict"] == "BEST_EFFORT_LOW_N")

# Provider counts table (actual usage at run time).
prov_counts_close = pregame_df["pregame_spread_provider"].value_counts(dropna=False)
prov_counts_open = pregame_df["opening_spread_provider"].value_counts(dropna=False)
prov_counts_ml = pregame_df["pregame_ml_provider"].value_counts(dropna=False)


def _provider_table(s: pd.Series) -> str:
    out = "| Provider | Games |\n|---|---:|\n"
    for k, v in s.items():
        k_disp = "(NULL)" if pd.isna(k) else str(k)
        out += f"| {k_disp} | {int(v):,} |\n"
    return out


def _column_doc_table() -> str:
    out = "| Column | Type | Unit | Range / sign | Category | Description |\n"
    out += "|---|---|---|---|---|---|\n"
    for name, typ, unit, rng, cat, desc in COLUMN_DOCS:
        out += f"| `{name}` | {typ} | {unit} | {rng} | {cat} | {desc} |\n"
    return out


now = time.strftime("%Y-%m-%d %H:%M:%S %Z").strip()
schema_md = f"""# trigger_events.csv — schema sidecar

**Generated:** {now}
**Source notebook:** `research/notebooks/01_trigger_events.ipynb`
**Source commit:** `{source_commit}`
**Trigger table version:** `{TRIGGER_TABLE_VERSION}`

This file documents the columns of `trigger_events.csv`, the provider priority
used at run time, and the empirical pre/post-play score-state findings that
the trigger detection function relies on. Every downstream notebook that
reads `trigger_events.csv` should treat this file as the canonical schema
reference. Do not paraphrase from memory.

## Column dictionary

Every column with type, unit, range/sign convention, and lookahead category.

**Categories:**
- **IDENTIFIER**: primary key components and within-game counters.
- **PRE-GAME CONTEXT**: observable before kickoff. R3-safe by definition.
- **POINT-IN-TIME GAME STATE**: derivable from plays with `playNumber <= trigger.play_number` only. R3-safe by construction (verified by `assert_no_lookahead`).

Game-final **LABEL** / target columns from the V5 DDL (`final_fav_won`, `final_fav_score`,
`final_dog_score`, `margin_of_defeat`) are **not** in this CSV — they live in the sibling
file `trigger_outcomes.csv` (same row order and natural key `(game_id, fav_deficit)`) so
feature notebooks (02a–g) never parse labels alongside in-game state. Notebook 03 joins
the two files explicitly. See `trigger_outcomes.schema.md`.

{_column_doc_table()}

## Provider priority used at run time

For pregame_spread, opening_spread, and pregame_fav_ml / pregame_dog_ml,
this notebook walks providers in priority order and takes the first non-null
value.

**Real-book priority (alphabetical):**

1. `Bovada`
2. `Caesars`
3. `DraftKings`
4. `ESPN Bet`

The N00 audit printed a per-(season, season_type) provider count matrix,
but that matrix was not written to a checked-in CSV, so per-provider
coverage cannot be cited here from project artifacts. Re-evaluate the
ordering in Notebook 03 if walk-forward results show ordering affects
calibration or CLV; the priority list is a top-of-notebook constant
(`PROVIDER_PRIORITY_REAL_BOOKS`) so reordering is a one-line change.

**Spread-only fallback:** `consensus`. Used when no real book has a spread
for that game/field. **Never used for moneylines** (consensus is a synthetic
mid-market estimate, not a tradeable price; using it for ML breaks devigging
math in N04).

**Excluded entirely:** `numberfire`, `teamrankings` (rating-service projections,
not market prices). State-specific Caesars variants
(`Caesars (Pennsylvania)`, etc.) are also excluded — only the canonical
`Caesars` entry is matched.

### Closing spread provider mix actually selected

{_provider_table(prov_counts_close)}

### Opening spread provider mix actually selected

{_provider_table(prov_counts_open)}

### Moneyline provider mix actually selected

{_provider_table(prov_counts_ml)}

## Score-state confidence levels

The trigger detection scorer (cell 16 of N01) does NOT assume a uniform
POST_PLAY or PRE_PLAY convention. Instead, every scoring play type that
may carry `scoring=true` is empirically verified against the cached
corpus (cell 14 of N01) and assigned to one of three confidence tiers.
The scorer branches on the tier; cell 14 raises if any subtype fails
verification, so triggers are never produced under a `FAIL` verdict.

Per-subtype verdicts also persist to `_subtype_verdicts.json` (sibling
file in this directory) so future N01 re-runs can diff and warn on
>2pp drift in any subtype's POST share.

### Tier 1 -- VERIFIED_POST_PLAY

>= {MIN_SAMPLES} samples in the cached corpus AND >= {int(VERIFIED_POST_THRESHOLD * 100)}% verified-POST
share. The scorer reads `offenseScore` / `defenseScore` direct, with no
cross-check. Standard guarantee.

**Subtypes ({_n_post_tier}):** {_tier_list("POST_PLAY")}

### Tier 2 -- POST_BEST_EFFORT (Kickoff Return TD carve-out)

>= {int(KO_RETURN_TD_THRESHOLD * 100)}% verified-POST share. The scorer reads direct, then
cross-checks against the same team's score at the next clean (non-PAT)
play with a +/-{KO_RETURN_TD_INVARIANT_TOLERANCE}-point tolerance. Cross-check failures are logged
to `SCORING_AMBIGUITY_LOG` and the play is skipped (no trigger emitted).
Documented carve-out for Kickoff Return TD, which has a small fraction
of pre-play recorded rows in the corpus.

**Subtypes ({_n_be_tier}):** {_tier_list("POST_BEST_EFFORT")}

### Tier 3 -- BEST_EFFORT_LOW_N

< {MIN_SAMPLES} samples in the cached corpus. The scorer reads direct and
applies a +/-{LOW_N_INVARIANT_TOLERANCE}-point cross-check at the next clean play. Soft
guarantee: with low sample counts the empirical verdict is statistically
weak. **Future N03 / N04 authors should treat trigger rows whose
`play_type` lower-cases to one of these subtypes with appropriate
skepticism**, or filter them out for the conservative variant of the
held-out test (`fav_deficit` count distribution by tier is reported in
this notebook's run summary, and `_subtype_verdicts.json` carries the
per-subtype sample sizes).

**Subtypes ({_n_low_tier}):** {_tier_list("BEST_EFFORT_LOW_N")}

### Empirical findings -- this run

{score_state_table}

**Convention applied to trigger detection:** `{SCORE_STATE_CONVENTION}`

The scorer's behavior is invariant to the `SCORE_STATE_CONVENTION`
banner; that constant is kept only for backward compatibility with
internal logging. The branching is by registry verdict per subtype.

## Generation provenance

- Notebook: `research/notebooks/01_trigger_events.ipynb`
- Commit hash: `{source_commit}`
- Generation timestamp: {now}
- Working-set games (post-exclusion, post-pickem): {len(pregame_df):,}
- Trigger rows emitted: {len(triggers_df_sorted):,}
- `trigger_events.csv` schema checks passed: 10/10
- `trigger_outcomes.csv` outcome consistency checks passed (see notebook)
- Spot-check sample size: {sample_size:,} rows ({sample_size / len(triggers_df) * 100:.1f}%)
- Spot-check discrepancies: 0
"""

SCHEMA_MD.write_text(schema_md, encoding="utf-8")
size_kb = SCHEMA_MD.stat().st_size / 1024
print(f"[ok] wrote {SCHEMA_MD.relative_to(REPO_ROOT)}: {size_kb:,.1f} KB")

# --- trigger_outcomes.schema.md (LABEL columns only) ------------------
OUTCOMES_COLUMN_DOCS: list[tuple[str, str, str, str, str, str]] = [
    ("game_id", "INTEGER", "—", "CFBD game ID", "IDENTIFIER / JOIN",
     "Natural key with `fav_deficit`. Must match `trigger_events.game_id`."),
    ("fav_deficit", "INTEGER", "points", "{3, 7, 10, 14, 21}", "IDENTIFIER / JOIN",
     "Natural key with `game_id`. Must match `trigger_events.fav_deficit`."),
    ("final_fav_won", "BOOLEAN", "—", "True | False | NULL", "LABEL",
     "Game-final outcome: True iff favorite's final score > underdog's after full game (including OT). R3-FORBIDDEN as a model **feature** — use only as prediction **target** in N03+ after an explicit join."),
    ("final_fav_score", "INTEGER", "points", ">=0 | NULL", "LABEL",
     "Favorite's final score. R3-FORBIDDEN as model feature."),
    ("final_dog_score", "INTEGER", "points", ">=0 | NULL", "LABEL",
     "Underdog's final score. R3-FORBIDDEN as model feature."),
    ("margin_of_defeat", "INTEGER", "points", ">0 | NULL", "LABEL",
     "If favorite lost: dog_final - fav_final (positive). NULL if favorite won or tie/unknown. R3-FORBIDDEN as model feature."),
]
_odoc_names = {c[0] for c in OUTCOMES_COLUMN_DOCS}
_actual_o = set(outcomes_df_sorted.columns)
assert _odoc_names == _actual_o, (
    f"Outcomes schema doc mismatch: {_odoc_names ^ _actual_o}"
)


def _outcomes_doc_table() -> str:
    out = "| Column | Type | Unit | Range / sign | Category | Description |\n"
    out += "|---|---|---|---|---|---|\n"
    for name, typ, unit, rng, cat, desc in OUTCOMES_COLUMN_DOCS:
        out += f"| `{name}` | {typ} | {unit} | {rng} | {cat} | {desc} |\n"
    return out


outcomes_schema_md = f"""# trigger_outcomes.csv — schema sidecar

**Generated:** {now}
**Source notebook:** `research/notebooks/01_trigger_events.ipynb`
**Source commit:** `{source_commit}`
**Trigger table version:** `{TRIGGER_TABLE_VERSION}`

This file documents the V5 **OUTCOME / LABEL** block that was **split out** of
`trigger_events.csv` for R3 structural safety. Feature notebooks load
`trigger_events.csv` **only**. Notebook 03 (and later) loads this file and joins
on `(game_id, fav_deficit)` after computing features — never merge labels before
feature extraction.

## Join contract

```python
df = trigger_events.merge(
    trigger_outcomes,
    on=["game_id", "fav_deficit"],
    how="inner",
    validate="one_to_one",
)
```

Row count equals `trigger_events.csv` row count (one outcome row per trigger row).

## Column dictionary

{_outcomes_doc_table()}

## Generation provenance

- Commit hash: `{source_commit}`
- Outcome rows: {len(outcomes_df_sorted):,}
"""

OUTCOMES_SCHEMA_MD.write_text(outcomes_schema_md, encoding="utf-8")
size_kb_o2 = OUTCOMES_SCHEMA_MD.stat().st_size / 1024
print(f"[ok] wrote {OUTCOMES_SCHEMA_MD.relative_to(REPO_ROOT)}: {size_kb_o2:,.1f} KB")

## Phase 0f — Summary, headline stats, budget print

In [ ]:
# --- Headline stats ---------------------------------------------------
print("=" * 64)
print("trigger_events build summary")
print("=" * 64)
n_games_in_corpus = len(games_df)
n_games_working = len(pregame_df)
n_games_with_trigger = triggers_df_sorted["game_id"].nunique()

print(f"\nCorpus:")
print(f"  total games (10 seasons, FBS-FBS, /games):   {n_games_in_corpus:,}")
print(f"  working set (post-exclusion + post-pickem):  {n_games_working:,}")
print(f"  games with at least one trigger:             {n_games_with_trigger:,}")
print(f"  games where favorite never trailed (regulation): {n_games_working - n_games_with_trigger:,}")

print(f"\nTrigger rows:")
print(f"  total: {len(triggers_df_sorted):,}")
print(f"  by deficit:")
for D, n in triggers_df_sorted["fav_deficit"].value_counts().sort_index().items():
    pct = n / len(triggers_df_sorted) * 100
    print(f"    D={D:>2}: {n:>5,} ({pct:5.1f}%)")
print(f"  by quarter:")
for q, n in triggers_df_sorted["quarter"].value_counts().sort_index().items():
    pct = n / len(triggers_df_sorted) * 100
    print(f"    Q{q}: {n:>5,} ({pct:5.1f}%)")

print(f"\nFavorite outcomes (R3 LABEL fields; do NOT use as model features):")
fw = outcomes_df_sorted["final_fav_won"]
n_won = int((fw == True).sum())  # noqa: E712
n_lost = int((fw == False).sum())  # noqa: E712
n_unknown = int(fw.isna().sum())
total = len(triggers_df_sorted)
print(f"  fav won (trigger rows):     {n_won:>5,} ({n_won/total*100:5.1f}%)")
print(f"  fav lost (trigger rows):    {n_lost:>5,} ({n_lost/total*100:5.1f}%)")
print(f"  unknown / tie:              {n_unknown:>5,} ({n_unknown/total*100:5.1f}%)")
print(f"  (trigger ROWS not games; multi-trigger games counted once per trigger)")

print(f"\nDeliverables (research/results/):")
for path in [TRIGGER_EVENTS_CSV, TRIGGER_OUTCOMES_CSV, BUCKETS_CSV, SCHEMA_MD, OUTCOMES_SCHEMA_MD]:
    size = path.stat().st_size
    print(f"  {path.name:<40} {size:>10,} bytes")

In [ ]:
# --- Final budget print ----------------------------------------------
calls_log_df = pd.read_csv(CALL_LOG)
n_total_log_rows = len(calls_log_df)
n_fresh_calls = int((calls_log_df["cached"] == 0).sum())
n_fresh_cfbd_total = int(((calls_log_df["service"] == "cfbd")
                          & (calls_log_df["cached"] == 0)).sum())

# Calls fresh THIS notebook run = calls logged after the start time of this run.
# We use a fuzzy proxy: the timestamp of the very first log line of this run
# is the time we printed cfbd_get setup. Easiest is to count fresh /plays and
# /drives, since N00 didn't pull either.
n_plays_fresh = int(((calls_log_df["service"] == "cfbd")
                     & (calls_log_df["endpoint"] == "/plays")
                     & (calls_log_df["cached"] == 0)).sum())
n_drives_fresh = int(((calls_log_df["service"] == "cfbd")
                      & (calls_log_df["endpoint"] == "/drives")
                      & (calls_log_df["cached"] == 0)).sum())
n_other_fresh_cfbd = n_fresh_cfbd_total - n_plays_fresh - n_drives_fresh

print("=" * 64)
print("CFBD call budget")
print("=" * 64)
print(f"\nThis notebook run (fresh CFBD calls):")
print(f"  /plays:  {n_plays_fresh:>3} fresh (budget: 162)")
print(f"  /drives: {n_drives_fresh:>3} fresh (budget: 20)")
print(f"  other:   {n_other_fresh_cfbd:>3} fresh (budget: 0 — should be 0; cache hits expected)")
print(f"  total fresh CFBD this run: {n_plays_fresh + n_drives_fresh + n_other_fresh_cfbd}")
print(f"  budget for this run: 182")

print(f"\nCumulative across all notebooks (call log: {n_total_log_rows:,} rows):")
print(f"  total fresh CFBD calls (lifetime): {n_fresh_cfbd_total:,}")
print(f"  monthly free-tier limit: 1,000")
print(f"  remaining this billing cycle: {1000 - n_fresh_cfbd_total:,}")
if n_fresh_cfbd_total >= 800:
    print(f"  [WARN] >=80% of monthly budget consumed.")
print(f"\n[ok] notebook 01 complete — STOP per R22. Do not start Notebook 02a without approval.")